# Part 2: Exploring Classification Algorithms

In [2]:
import pandas as pd
from IPython.display import display
from IPython.display import Markdown
from sklearn.decomposition import PCA
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import r_regression
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import make_scorer
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import label_binarize
from sklearn.tree import DecisionTreeClassifier


In [3]:
def report(string):
    display(Markdown(string))

class Table:
    def tableHeader(self):
        doc = "| Algorithm | Hyperparam | Value | accuracy | precision | recall | f1 |\n"
        doc += "| --- | --- | --- | --- |--- | --- | --- |\n"
        return doc

    def __init__(self):
        self.prev_algo = None
        self.prev_hyper = None
        self.prev_value = None
        self.doc = self.tableHeader()

    def tableCVScore(self, algorithm=None, hyperparameter=None, value=None, scores=None):
        rows = ""
        accuracy = scores['test_accuracy']
        precision = scores['test_precision']
        recall = scores['test_recall']
        f1 = scores['test_f1_score']
        out_algo = ""
        out_hyper = ""
        if (self.prev_algo != algorithm):
            self.prev_algo = algorithm
            out_algo = algorithm
            out_hyper = hyperparameter
        if (self.prev_hyper != hyperparameter):
            self.prev_hyper = hyperparameter
            out_hyper = hyperparameter
        rows += f"| {out_algo} | {out_hyper} | {value} | {accuracy:.4g} | {precision:.4g} | {recall:.4g} | {f1:.4g} |\n"
        self.doc = self.doc + rows
        return self

    def tableFooter(self):
        report(self.doc)


enable_debug = False

def debug(string):
    if (enable_debug):
        print(string)


# https://xkcd.com/221/ - 4 is overused
random_seed = 221

In [4]:
cross_validation_scores = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, average='weighted'),
    'recall': make_scorer(recall_score, average='weighted'),
    'f1_score': make_scorer(f1_score, average='weighted')
}

def crossValidateUsingTree(random_state=random_seed, table=None, algorithm=None, hyperparameter=None, value=None, X=None, Y=None, max_depth=None,
                           cv=5):
    tree = DecisionTreeClassifier(criterion="entropy", random_state=random_state, max_depth=max_depth)
    cv_scores = cross_validate(tree, X, Y, cv=cv, scoring=cross_validation_scores)
    cv_scores_df = pd.DataFrame(cv_scores)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=cv_scores_df.mean())
    return cv_scores_df

def crossValidateUsingAdaBoost(random_state=random_seed, table=None, algorithm=None, hyperparameter=None, value=None, X=None, Y=None, n_estimators=50,
                               cv=5):
    tree = AdaBoostClassifier(random_state=random_state, n_estimators=n_estimators)
    cv_scores = cross_validate(tree, X, Y[target_column], cv=cv, scoring=cross_validation_scores)
    cv_scores_df = pd.DataFrame(cv_scores)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=cv_scores_df.mean())
    return cv_scores_df

def crossValidateUsingRandomForest(random_state=random_seed, table=None, algorithm=None, hyperparameter=None, value=None, X=None, Y=None, n_estimators=100,
                                   cv=5):
    tree = RandomForestClassifier(criterion="entropy", random_state=random_state, n_estimators=n_estimators)
    cv_scores = cross_validate(tree, X, Y[target_column], cv=cv, scoring=cross_validation_scores)
    cv_scores_df = pd.DataFrame(cv_scores)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=cv_scores_df.mean())
    return cv_scores_df

def crossValidateUsingKnn(table=None, X=None, Y=None, algorithm=None, hyperparameter=None, value=None, n_neighbours=5,
                          cv=5):
    knn = KNeighborsClassifier(n_neighbors=n_neighbours)
    cv_scores = cross_validate(knn, X, Y[target_column], cv=cv, scoring=cross_validation_scores)
    cv_scores_df = pd.DataFrame(cv_scores)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=cv_scores_df.mean())
    return cv_scores_df

def scores(Y, Y_pred):
    return {
        'test_accuracy': accuracy_score(Y, Y_pred),
        'test_precision': precision_score(Y, Y_pred, average='weighted'),
        'test_recall': recall_score(Y, Y_pred, average='weighted'),
        'test_f1_score': f1_score(Y, Y_pred, average='weighted'),
    }

def trainAndPredictDecisionTree(random_state=random_seed, table=None, algorithm=None, hyperparameter=None, value=None, X=None, Y=None, max_depth=None, testX=None, testY=None):
    tree = DecisionTreeClassifier(criterion="entropy", random_state=random_state, max_depth=max_depth)
    tree.fit(X, Y)
    y_pred = tree.predict(testX)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=scores(testY, y_pred))
    return table

def trainAndPredictAdaBoost(random_state=random_seed, table=None, algorithm=None, hyperparameter=None, value=None, X=None, Y=None, n_estimators=50, testX=None, testY=None):
    tree = AdaBoostClassifier(random_state=random_state, n_estimators=n_estimators)
    tree.fit(X,Y[target_column])
    y_pred = tree.predict(testX)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=scores(testY, y_pred))
    return table

def trainAndPredictRandomForest(random_state=random_seed, table=None, algorithm=None, hyperparameter=None, value=None, X=None, Y=None, n_estimators=100, testX=None, testY=None):
    tree = RandomForestClassifier(criterion="entropy", random_state=random_state, n_estimators=n_estimators)
    tree.fit(X, Y[target_column])
    y_pred = tree.predict(testX)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=scores(testY, y_pred))
    return table

def trainAndPredictKnn(table=None, X=None, Y=None, algorithm=None, hyperparameter=None, value=None, n_neighbours=5, testX=None, testY=None):
    knn = KNeighborsClassifier(n_neighbors=n_neighbours)
    knn.fit(X, Y[target_column])
    y_pred = knn.predict(testX)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=scores(testY, y_pred))
    return table


## Data Loading

In [5]:
df_csv = pd.read_csv('../data/breast-cancer.csv')

target_column = 'diagnosis'
id_column = 'id'

feature_names = df_csv.columns.tolist()
debug(feature_names)
df_csv = df_csv.drop(id_column, axis=1)

Y_df = df_csv[[target_column]]
X_df = df_csv.drop([target_column], axis=1)
# unique converts to an NDArray, so the sort needs to happen first.
target_columns = Y_df.columns.tolist()
debug(f"target column: {target_columns}")
feature_names = X_df.columns.tolist()
debug(f"feature names: {feature_names}")
labels = Y_df[target_column].sort_values().unique()
debug(f"labels: {labels}")

# Split the data into Train and Test
X_train, X_test, Y_train, Y_test = train_test_split(
    X_df, Y_df, test_size= 0.3, random_state=random_seed)

## 1. Missing Data Processing and Data Normalization

In [6]:
mean_imputer = SimpleImputer(missing_values=0, strategy='mean')
mean_imputer.fit(X_train)
X_train_mean = mean_imputer.transform(X_train)
X_test_mean = mean_imputer.transform(X_test)
X_df_mean = mean_imputer.transform(X_df)

median_imputer = SimpleImputer(missing_values=0, strategy='median')
median_imputer.fit(X_train)
X_train_median = median_imputer.transform(X_train)
X_test_median = median_imputer.transform(X_test)
X_df_median = median_imputer.transform(X_df)

## 1.a DecisionTree evaluation of imputation methods

In [7]:
table = Table()

parameters = {
  "table":table, "algorithm": "Decision Tree", "hyperparameter": "imputation"
}
crossValidateUsingTree(**parameters, value="baseline", X=X_train, Y=Y_train)
crossValidateUsingTree(**parameters, value="mean", X=X_train_mean, Y=Y_train)
crossValidateUsingTree(**parameters, value="median", X=X_train_median, Y=Y_train)
crossValidateUsingTree(**parameters, value="full mean", X=X_df_mean, Y=Y_df)
crossValidateUsingTree(**parameters, value="full median", X=X_df_median, Y=Y_df)

table.tableFooter()


| Algorithm | Hyperparam | Value | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- | --- |
| Decision Tree | imputation | baseline | 0.9398 | 0.9419 | 0.9398 | 0.9396 |
|  |  | mean | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
|  |  | median | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
|  |  | full mean | 0.9244 | 0.9276 | 0.9244 | 0.9242 |
|  |  | full median | 0.9262 | 0.929 | 0.9262 | 0.9259 |


### Evaluation of imputation methods

For evaluation, the evaluation metric was set to "entropy" for information gain, and max_depth is None to give the tree the best chance at success. Decision Trees are very tolerant of missing data. When using cross validation, both median and mean imputation produced a tree that performed worse than baseline.

Adding in median and mean cross validation using the full dataset, we see that median works slightly better than mean, which for a decision tree would make sense. It puts the missing value in the middle of the data, ignoring outliers.

In [8]:
X_train = X_train_median
X_test = X_test_median

## 1.b Standardization

In [9]:
standard_scaler = StandardScaler()
standard_scaler.fit(X_train)
X_train_standardized = standard_scaler.transform(X_train, copy=True)
X_test_standardized = standard_scaler.transform(X_test, copy=True)
X_df_standardized = standard_scaler.transform(X_df_median, copy=True)

min_max_scaler = MinMaxScaler(copy=True) # don't clip. Why is MinMaxScaler different to StandardizedScaler?!?
min_max_scaler.fit(X_train)
X_train_min_max = min_max_scaler.transform(X_train)
X_test_min_max = min_max_scaler.transform(X_test)
X_df_min_max = min_max_scaler.transform(X_df_median)


In [10]:
table = Table()

parameters = {
    "table":table,
    "algorithm":"Decision Tree", "hyperparameter":"normalization"
}
crossValidateUsingTree(**parameters, value="baseline", X=X_train, Y=Y_train)
crossValidateUsingTree(**parameters, value="standardized", X=X_train_standardized, Y=Y_train)
crossValidateUsingTree(**parameters, value="min_max", X=X_train_min_max, Y=Y_train)
crossValidateUsingTree(**parameters, value="full standardized", X=X_df_standardized, Y=Y_df)
crossValidateUsingTree(**parameters, value="full min_max", X=X_df_min_max, Y=Y_df)

parameters = {
    "table":table, "algorithm": "KNN", "hyperparameter": "normalization"
}
crossValidateUsingKnn(**parameters, value="baseline", X=X_train, Y=Y_train)
crossValidateUsingKnn(**parameters, value="standardized", X=X_train_standardized, Y=Y_train)
crossValidateUsingKnn(**parameters, value="min-max", X=X_train_min_max, Y=Y_train)
crossValidateUsingKnn(**parameters, value="full standardized", X=X_df_standardized, Y=Y_df)
crossValidateUsingKnn(**parameters, value="full min-max", X=X_df_min_max, Y=Y_df)

table.tableFooter()

| Algorithm | Hyperparam | Value | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- | --- |
| Decision Tree | normalization | baseline | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
|  |  | standardized | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
|  |  | min_max | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
|  |  | full standardized | 0.9262 | 0.929 | 0.9262 | 0.9259 |
|  |  | full min_max | 0.9262 | 0.929 | 0.9262 | 0.9259 |
| KNN | normalization | baseline | 0.9373 | 0.9385 | 0.9373 | 0.9362 |
|  |  | standardized | 0.9598 | 0.9614 | 0.9598 | 0.9593 |
|  |  | min-max | 0.9624 | 0.9637 | 0.9624 | 0.9619 |
|  |  | full standardized | 0.9666 | 0.9675 | 0.9666 | 0.9664 |
|  |  | full min-max | 0.9701 | 0.9706 | 0.9701 | 0.97 |


### Evaluation of normalization methods

There is no benefit to standardization over baseline in decision trees for either method. This makes sense because the tree is splitting at a threshold, so the number of thresholds doesn't change with scale changes.

KNN shows an improvement using min-max, and this continues through to the full dataset. This implies that the data isn't normally distributed, with multiple modes. Standardization provides additional bits to the data values close to the mean. MinMax scaling provides the same number of bits over the address range. If there are multiple modes at either end of the distribution, MinMax will allow them to be more easily differentiated, where standardization might lose too many digits.

In [11]:
X_train = X_train_min_max
X_test = X_test_min_max

## 2. Classifier Exploration and Hyperparameter Tuning

In [12]:
table = Table()

parameters = {
  "table":table, "algorithm": "KNN", "hyperparameter": "n_neighbours",
  "X":X_train, "Y":Y_train, "testX" : X_test, "testY": Y_test
}
knn_neighbours = [3,9,15,21]
for n in knn_neighbours:
    trainAndPredictKnn(**parameters, value=n, n_neighbours=n)

parameters = {
    "table":table, "algorithm": "Decision Tree", "hyperparameter": "max_depth",
    "X":X_train, "Y":Y_train, "testX" : X_test, "testY": Y_test
}
decision_tree_depths = [2,8,14,None]
for n in decision_tree_depths:
    trainAndPredictDecisionTree(**parameters, value=n, max_depth=n)

parameters = {
    "table":table, "algorithm": "AdaBoost", "hyperparameter": "n_estimators",
    "X":X_train, "Y":Y_train, "testX" : X_test, "testY": Y_test
}
n_estimators = [10,20,30,100]
for n in n_estimators:
    trainAndPredictAdaBoost(**parameters, value=n, n_estimators=n)

parameters = {
    "table":table, "algorithm": "Random Forest", "hyperparameter": "n_estimators",
    "X":X_train, "Y":Y_train, "testX" : X_test, "testY": Y_test
}
n_estimators = [10,30,50,60,100]
for n in n_estimators:
    trainAndPredictRandomForest(**parameters, value=n, n_estimators=n)

table.tableFooter()

| Algorithm | Hyperparam | Value | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- | --- |
| KNN | n_neighbours | 3 | 0.9649 | 0.9668 | 0.9649 | 0.9646 |
|  |  | 9 | 0.9708 | 0.9721 | 0.9708 | 0.9705 |
|  |  | 15 | 0.9708 | 0.9721 | 0.9708 | 0.9705 |
|  |  | 21 | 0.9532 | 0.9566 | 0.9532 | 0.9526 |
| Decision Tree | max_depth | 2 | 0.883 | 0.8841 | 0.883 | 0.8815 |
|  |  | 8 | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
|  |  | 14 | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
|  |  | None | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
| AdaBoost | n_estimators | 10 | 0.9591 | 0.9602 | 0.9591 | 0.9588 |
|  |  | 20 | 0.9532 | 0.9549 | 0.9532 | 0.9528 |
|  |  | 30 | 0.9649 | 0.9656 | 0.9649 | 0.9647 |
|  |  | 100 | 0.9649 | 0.9656 | 0.9649 | 0.9647 |
| Random Forest | n_estimators | 10 | 0.9298 | 0.931 | 0.9298 | 0.9292 |
|  |  | 30 | 0.9474 | 0.9483 | 0.9474 | 0.947 |
|  |  | 50 | 0.9649 | 0.9668 | 0.9649 | 0.9646 |
|  |  | 60 | 0.9591 | 0.9616 | 0.9591 | 0.9586 |
|  |  | 100 | 0.9532 | 0.9566 | 0.9532 | 0.9526 |


### Observe and discuss the impact of varying the hyperparameter

This felt a little dirty, evaluating the hyperparameter against the test data. We can explain the results afterwards, but predicting them is difficult. All algorithms other than KNN increased in performance as the hyperparameter was increased. This makes sense as the non-KNN algorithms will add more information as the parameter is increased. I added N=100 and max_depth=None to see if there were elbows where performance dropped off. Changing the hyperparameters had very small impacts on overall performance, less than 1% for KNN, 2% for AdaBoost, 3.5% for Random Forest. Decision Tree showed the largest swing at 6%.

AdaBoost performed best at N=30. AdaBoost learns which instances are classified improperly, weighting them in the next iteration. As more iterations are performed, the error term will naturally decrease. The algorithm is able to continue to decrease the error as more cpu is expended. However, each iteration will also add another decision tree to the final model.

RandomForest was second best with n = 50, outperforming the regular decision tree. The random subset used for each tree allow features which are unimportant to have a chance to be ignored. However, as the number of trees increases, performance dropped back down.

Decision Tree hit peak performance at max_depth=8. This is likely because there were no further splits in the training data. Given the 539 instances, a 70% training split = 377, which is smaller than the size of a fully populated tree at 2^8 * 2 = 512 nodes. The minimum leaf size won't allow more nodes. Decision Trees do seem to have a minimum required number of nodes, somwhere between 10 and 20 for this dataset.

KNN peaked at n = 3. Increasing the number of neighbours decreased performance. This is likely because the correct n to use is related to the number of instances in the dataset, and the number of expected clusters.

This shows that algorithms that are able to improve their error term with additional iterations will outperform algorithms that increase the amount of data considered.

## 3. Feature Engineering and Dimensionality Reduction

### 3.a Pearson Correlation

In [42]:
# Convert back to DataFrame so we can see what's what.
report("**Step 1 - use normalised data**")
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

report("**Step 2 - translate Y to binary**")
Y_train_transformed = label_binarize(Y_train[target_column].tolist(), classes=['B', 'M']).ravel()
Y_test_transformed = label_binarize(Y_test[target_column].tolist(), classes=['B', 'M']).ravel()

report("**Step 3 - run the regression**")
pearson = r_regression(X_train, Y_train_transformed)

report("**Step 4 - Iterate over the list, and find the indices that are > 0.6**")
X_train_filtered = X_train_df.copy()
X_test_filtered = X_test_df.copy()
threshold = 0.6

pearson_filtered_columns = []
for i in range(0, len(pearson)):
    r = pearson[i]
    if r >= threshold:
        pearson_filtered_columns.append(i)

X_train_filtered = X_train_filtered.iloc[:,pearson_filtered_columns]
X_test_filtered = X_test_filtered.iloc[:,pearson_filtered_columns]

report("**Step 5 - Produce output for report**")
report("**Step 5.a - Generate the 2d correlation matrix using Pandas - otherwise unused**")
X_correlation = X_train_df.copy()
X_correlation[target_column] = Y_train_transformed

two_d_correlation = X_correlation.corr(method="pearson")

# Shrink the table down to hopefully render in the PDF.
styled_df = (two_d_correlation.style
             .format(precision=2)
             .set_table_styles(
               [{'selector': 'th', 'props': [('font-size', '5px')]}])
             .set_properties(**{'font-size': '6pt', 'font-family': 'Arial'})
             )
report("**Correlation Matrix:**")
report(styled_df.to_html())

report(f"**Number of original features:** {len(feature_names)}")
report(f"**Number retained features:** {len(X_train_filtered.columns)}")

**Step 1 - use normalised data**

**Step 2 - translate Y to binary**

**Step 3 - run the regression**

**Step 4 - Iterate over the list, and find the indices that are > 0.6**

**Step 5 - Produce output for report**

**Step 5.a - Generate the 2d correlation matrix using Pandas - otherwise unused**

**Correlation Matrix:**

<style type="text/css">
#T_4edac th {
  font-size: 5px;
}
#T_4edac_row0_col0, #T_4edac_row0_col1, #T_4edac_row0_col2, #T_4edac_row0_col3, #T_4edac_row0_col4, #T_4edac_row0_col5, #T_4edac_row0_col6, #T_4edac_row0_col7, #T_4edac_row0_col8, #T_4edac_row0_col9, #T_4edac_row0_col10, #T_4edac_row0_col11, #T_4edac_row0_col12, #T_4edac_row0_col13, #T_4edac_row0_col14, #T_4edac_row0_col15, #T_4edac_row0_col16, #T_4edac_row0_col17, #T_4edac_row0_col18, #T_4edac_row0_col19, #T_4edac_row0_col20, #T_4edac_row0_col21, #T_4edac_row0_col22, #T_4edac_row0_col23, #T_4edac_row0_col24, #T_4edac_row0_col25, #T_4edac_row0_col26, #T_4edac_row0_col27, #T_4edac_row0_col28, #T_4edac_row0_col29, #T_4edac_row0_col30, #T_4edac_row1_col0, #T_4edac_row1_col1, #T_4edac_row1_col2, #T_4edac_row1_col3, #T_4edac_row1_col4, #T_4edac_row1_col5, #T_4edac_row1_col6, #T_4edac_row1_col7, #T_4edac_row1_col8, #T_4edac_row1_col9, #T_4edac_row1_col10, #T_4edac_row1_col11, #T_4edac_row1_col12, #T_4edac_row1_col13, #T_4edac_row1_col14, #T_4edac_row1_col15, #T_4edac_row1_col16, #T_4edac_row1_col17, #T_4edac_row1_col18, #T_4edac_row1_col19, #T_4edac_row1_col20, #T_4edac_row1_col21, #T_4edac_row1_col22, #T_4edac_row1_col23, #T_4edac_row1_col24, #T_4edac_row1_col25, #T_4edac_row1_col26, #T_4edac_row1_col27, #T_4edac_row1_col28, #T_4edac_row1_col29, #T_4edac_row1_col30, #T_4edac_row2_col0, #T_4edac_row2_col1, #T_4edac_row2_col2, #T_4edac_row2_col3, #T_4edac_row2_col4, #T_4edac_row2_col5, #T_4edac_row2_col6, #T_4edac_row2_col7, #T_4edac_row2_col8, #T_4edac_row2_col9, #T_4edac_row2_col10, #T_4edac_row2_col11, #T_4edac_row2_col12, #T_4edac_row2_col13, #T_4edac_row2_col14, #T_4edac_row2_col15, #T_4edac_row2_col16, #T_4edac_row2_col17, #T_4edac_row2_col18, #T_4edac_row2_col19, #T_4edac_row2_col20, #T_4edac_row2_col21, #T_4edac_row2_col22, #T_4edac_row2_col23, #T_4edac_row2_col24, #T_4edac_row2_col25, #T_4edac_row2_col26, #T_4edac_row2_col27, #T_4edac_row2_col28, #T_4edac_row2_col29, #T_4edac_row2_col30, #T_4edac_row3_col0, #T_4edac_row3_col1, #T_4edac_row3_col2, #T_4edac_row3_col3, #T_4edac_row3_col4, #T_4edac_row3_col5, #T_4edac_row3_col6, #T_4edac_row3_col7, #T_4edac_row3_col8, #T_4edac_row3_col9, #T_4edac_row3_col10, #T_4edac_row3_col11, #T_4edac_row3_col12, #T_4edac_row3_col13, #T_4edac_row3_col14, #T_4edac_row3_col15, #T_4edac_row3_col16, #T_4edac_row3_col17, #T_4edac_row3_col18, #T_4edac_row3_col19, #T_4edac_row3_col20, #T_4edac_row3_col21, #T_4edac_row3_col22, #T_4edac_row3_col23, #T_4edac_row3_col24, #T_4edac_row3_col25, #T_4edac_row3_col26, #T_4edac_row3_col27, #T_4edac_row3_col28, #T_4edac_row3_col29, #T_4edac_row3_col30, #T_4edac_row4_col0, #T_4edac_row4_col1, #T_4edac_row4_col2, #T_4edac_row4_col3, #T_4edac_row4_col4, #T_4edac_row4_col5, #T_4edac_row4_col6, #T_4edac_row4_col7, #T_4edac_row4_col8, #T_4edac_row4_col9, #T_4edac_row4_col10, #T_4edac_row4_col11, #T_4edac_row4_col12, #T_4edac_row4_col13, #T_4edac_row4_col14, #T_4edac_row4_col15, #T_4edac_row4_col16, #T_4edac_row4_col17, #T_4edac_row4_col18, #T_4edac_row4_col19, #T_4edac_row4_col20, #T_4edac_row4_col21, #T_4edac_row4_col22, #T_4edac_row4_col23, #T_4edac_row4_col24, #T_4edac_row4_col25, #T_4edac_row4_col26, #T_4edac_row4_col27, #T_4edac_row4_col28, #T_4edac_row4_col29, #T_4edac_row4_col30, #T_4edac_row5_col0, #T_4edac_row5_col1, #T_4edac_row5_col2, #T_4edac_row5_col3, #T_4edac_row5_col4, #T_4edac_row5_col5, #T_4edac_row5_col6, #T_4edac_row5_col7, #T_4edac_row5_col8, #T_4edac_row5_col9, #T_4edac_row5_col10, #T_4edac_row5_col11, #T_4edac_row5_col12, #T_4edac_row5_col13, #T_4edac_row5_col14, #T_4edac_row5_col15, #T_4edac_row5_col16, #T_4edac_row5_col17, #T_4edac_row5_col18, #T_4edac_row5_col19, #T_4edac_row5_col20, #T_4edac_row5_col21, #T_4edac_row5_col22, #T_4edac_row5_col23, #T_4edac_row5_col24, #T_4edac_row5_col25, #T_4edac_row5_col26, #T_4edac_row5_col27, #T_4edac_row5_col28, #T_4edac_row5_col29, #T_4edac_row5_col30, #T_4edac_row6_col0, #T_4edac_row6_col1, #T_4edac_row6_col2, #T_4edac_row6_col3, #T_4edac_row6_col4, #T_4edac_row6_col5, #T_4edac_row6_col6, #T_4edac_row6_col7, #T_4edac_row6_col8, #T_4edac_row6_col9, #T_4edac_row6_col10, #T_4edac_row6_col11, #T_4edac_row6_col12, #T_4edac_row6_col13, #T_4edac_row6_col14, #T_4edac_row6_col15, #T_4edac_row6_col16, #T_4edac_row6_col17, #T_4edac_row6_col18, #T_4edac_row6_col19, #T_4edac_row6_col20, #T_4edac_row6_col21, #T_4edac_row6_col22, #T_4edac_row6_col23, #T_4edac_row6_col24, #T_4edac_row6_col25, #T_4edac_row6_col26, #T_4edac_row6_col27, #T_4edac_row6_col28, #T_4edac_row6_col29, #T_4edac_row6_col30, #T_4edac_row7_col0, #T_4edac_row7_col1, #T_4edac_row7_col2, #T_4edac_row7_col3, #T_4edac_row7_col4, #T_4edac_row7_col5, #T_4edac_row7_col6, #T_4edac_row7_col7, #T_4edac_row7_col8, #T_4edac_row7_col9, #T_4edac_row7_col10, #T_4edac_row7_col11, #T_4edac_row7_col12, #T_4edac_row7_col13, #T_4edac_row7_col14, #T_4edac_row7_col15, #T_4edac_row7_col16, #T_4edac_row7_col17, #T_4edac_row7_col18, #T_4edac_row7_col19, #T_4edac_row7_col20, #T_4edac_row7_col21, #T_4edac_row7_col22, #T_4edac_row7_col23, #T_4edac_row7_col24, #T_4edac_row7_col25, #T_4edac_row7_col26, #T_4edac_row7_col27, #T_4edac_row7_col28, #T_4edac_row7_col29, #T_4edac_row7_col30, #T_4edac_row8_col0, #T_4edac_row8_col1, #T_4edac_row8_col2, #T_4edac_row8_col3, #T_4edac_row8_col4, #T_4edac_row8_col5, #T_4edac_row8_col6, #T_4edac_row8_col7, #T_4edac_row8_col8, #T_4edac_row8_col9, #T_4edac_row8_col10, #T_4edac_row8_col11, #T_4edac_row8_col12, #T_4edac_row8_col13, #T_4edac_row8_col14, #T_4edac_row8_col15, #T_4edac_row8_col16, #T_4edac_row8_col17, #T_4edac_row8_col18, #T_4edac_row8_col19, #T_4edac_row8_col20, #T_4edac_row8_col21, #T_4edac_row8_col22, #T_4edac_row8_col23, #T_4edac_row8_col24, #T_4edac_row8_col25, #T_4edac_row8_col26, #T_4edac_row8_col27, #T_4edac_row8_col28, #T_4edac_row8_col29, #T_4edac_row8_col30, #T_4edac_row9_col0, #T_4edac_row9_col1, #T_4edac_row9_col2, #T_4edac_row9_col3, #T_4edac_row9_col4, #T_4edac_row9_col5, #T_4edac_row9_col6, #T_4edac_row9_col7, #T_4edac_row9_col8, #T_4edac_row9_col9, #T_4edac_row9_col10, #T_4edac_row9_col11, #T_4edac_row9_col12, #T_4edac_row9_col13, #T_4edac_row9_col14, #T_4edac_row9_col15, #T_4edac_row9_col16, #T_4edac_row9_col17, #T_4edac_row9_col18, #T_4edac_row9_col19, #T_4edac_row9_col20, #T_4edac_row9_col21, #T_4edac_row9_col22, #T_4edac_row9_col23, #T_4edac_row9_col24, #T_4edac_row9_col25, #T_4edac_row9_col26, #T_4edac_row9_col27, #T_4edac_row9_col28, #T_4edac_row9_col29, #T_4edac_row9_col30, #T_4edac_row10_col0, #T_4edac_row10_col1, #T_4edac_row10_col2, #T_4edac_row10_col3, #T_4edac_row10_col4, #T_4edac_row10_col5, #T_4edac_row10_col6, #T_4edac_row10_col7, #T_4edac_row10_col8, #T_4edac_row10_col9, #T_4edac_row10_col10, #T_4edac_row10_col11, #T_4edac_row10_col12, #T_4edac_row10_col13, #T_4edac_row10_col14, #T_4edac_row10_col15, #T_4edac_row10_col16, #T_4edac_row10_col17, #T_4edac_row10_col18, #T_4edac_row10_col19, #T_4edac_row10_col20, #T_4edac_row10_col21, #T_4edac_row10_col22, #T_4edac_row10_col23, #T_4edac_row10_col24, #T_4edac_row10_col25, #T_4edac_row10_col26, #T_4edac_row10_col27, #T_4edac_row10_col28, #T_4edac_row10_col29, #T_4edac_row10_col30, #T_4edac_row11_col0, #T_4edac_row11_col1, #T_4edac_row11_col2, #T_4edac_row11_col3, #T_4edac_row11_col4, #T_4edac_row11_col5, #T_4edac_row11_col6, #T_4edac_row11_col7, #T_4edac_row11_col8, #T_4edac_row11_col9, #T_4edac_row11_col10, #T_4edac_row11_col11, #T_4edac_row11_col12, #T_4edac_row11_col13, #T_4edac_row11_col14, #T_4edac_row11_col15, #T_4edac_row11_col16, #T_4edac_row11_col17, #T_4edac_row11_col18, #T_4edac_row11_col19, #T_4edac_row11_col20, #T_4edac_row11_col21, #T_4edac_row11_col22, #T_4edac_row11_col23, #T_4edac_row11_col24, #T_4edac_row11_col25, #T_4edac_row11_col26, #T_4edac_row11_col27, #T_4edac_row11_col28, #T_4edac_row11_col29, #T_4edac_row11_col30, #T_4edac_row12_col0, #T_4edac_row12_col1, #T_4edac_row12_col2, #T_4edac_row12_col3, #T_4edac_row12_col4, #T_4edac_row12_col5, #T_4edac_row12_col6, #T_4edac_row12_col7, #T_4edac_row12_col8, #T_4edac_row12_col9, #T_4edac_row12_col10, #T_4edac_row12_col11, #T_4edac_row12_col12, #T_4edac_row12_col13, #T_4edac_row12_col14, #T_4edac_row12_col15, #T_4edac_row12_col16, #T_4edac_row12_col17, #T_4edac_row12_col18, #T_4edac_row12_col19, #T_4edac_row12_col20, #T_4edac_row12_col21, #T_4edac_row12_col22, #T_4edac_row12_col23, #T_4edac_row12_col24, #T_4edac_row12_col25, #T_4edac_row12_col26, #T_4edac_row12_col27, #T_4edac_row12_col28, #T_4edac_row12_col29, #T_4edac_row12_col30, #T_4edac_row13_col0, #T_4edac_row13_col1, #T_4edac_row13_col2, #T_4edac_row13_col3, #T_4edac_row13_col4, #T_4edac_row13_col5, #T_4edac_row13_col6, #T_4edac_row13_col7, #T_4edac_row13_col8, #T_4edac_row13_col9, #T_4edac_row13_col10, #T_4edac_row13_col11, #T_4edac_row13_col12, #T_4edac_row13_col13, #T_4edac_row13_col14, #T_4edac_row13_col15, #T_4edac_row13_col16, #T_4edac_row13_col17, #T_4edac_row13_col18, #T_4edac_row13_col19, #T_4edac_row13_col20, #T_4edac_row13_col21, #T_4edac_row13_col22, #T_4edac_row13_col23, #T_4edac_row13_col24, #T_4edac_row13_col25, #T_4edac_row13_col26, #T_4edac_row13_col27, #T_4edac_row13_col28, #T_4edac_row13_col29, #T_4edac_row13_col30, #T_4edac_row14_col0, #T_4edac_row14_col1, #T_4edac_row14_col2, #T_4edac_row14_col3, #T_4edac_row14_col4, #T_4edac_row14_col5, #T_4edac_row14_col6, #T_4edac_row14_col7, #T_4edac_row14_col8, #T_4edac_row14_col9, #T_4edac_row14_col10, #T_4edac_row14_col11, #T_4edac_row14_col12, #T_4edac_row14_col13, #T_4edac_row14_col14, #T_4edac_row14_col15, #T_4edac_row14_col16, #T_4edac_row14_col17, #T_4edac_row14_col18, #T_4edac_row14_col19, #T_4edac_row14_col20, #T_4edac_row14_col21, #T_4edac_row14_col22, #T_4edac_row14_col23, #T_4edac_row14_col24, #T_4edac_row14_col25, #T_4edac_row14_col26, #T_4edac_row14_col27, #T_4edac_row14_col28, #T_4edac_row14_col29, #T_4edac_row14_col30, #T_4edac_row15_col0, #T_4edac_row15_col1, #T_4edac_row15_col2, #T_4edac_row15_col3, #T_4edac_row15_col4, #T_4edac_row15_col5, #T_4edac_row15_col6, #T_4edac_row15_col7, #T_4edac_row15_col8, #T_4edac_row15_col9, #T_4edac_row15_col10, #T_4edac_row15_col11, #T_4edac_row15_col12, #T_4edac_row15_col13, #T_4edac_row15_col14, #T_4edac_row15_col15, #T_4edac_row15_col16, #T_4edac_row15_col17, #T_4edac_row15_col18, #T_4edac_row15_col19, #T_4edac_row15_col20, #T_4edac_row15_col21, #T_4edac_row15_col22, #T_4edac_row15_col23, #T_4edac_row15_col24, #T_4edac_row15_col25, #T_4edac_row15_col26, #T_4edac_row15_col27, #T_4edac_row15_col28, #T_4edac_row15_col29, #T_4edac_row15_col30, #T_4edac_row16_col0, #T_4edac_row16_col1, #T_4edac_row16_col2, #T_4edac_row16_col3, #T_4edac_row16_col4, #T_4edac_row16_col5, #T_4edac_row16_col6, #T_4edac_row16_col7, #T_4edac_row16_col8, #T_4edac_row16_col9, #T_4edac_row16_col10, #T_4edac_row16_col11, #T_4edac_row16_col12, #T_4edac_row16_col13, #T_4edac_row16_col14, #T_4edac_row16_col15, #T_4edac_row16_col16, #T_4edac_row16_col17, #T_4edac_row16_col18, #T_4edac_row16_col19, #T_4edac_row16_col20, #T_4edac_row16_col21, #T_4edac_row16_col22, #T_4edac_row16_col23, #T_4edac_row16_col24, #T_4edac_row16_col25, #T_4edac_row16_col26, #T_4edac_row16_col27, #T_4edac_row16_col28, #T_4edac_row16_col29, #T_4edac_row16_col30, #T_4edac_row17_col0, #T_4edac_row17_col1, #T_4edac_row17_col2, #T_4edac_row17_col3, #T_4edac_row17_col4, #T_4edac_row17_col5, #T_4edac_row17_col6, #T_4edac_row17_col7, #T_4edac_row17_col8, #T_4edac_row17_col9, #T_4edac_row17_col10, #T_4edac_row17_col11, #T_4edac_row17_col12, #T_4edac_row17_col13, #T_4edac_row17_col14, #T_4edac_row17_col15, #T_4edac_row17_col16, #T_4edac_row17_col17, #T_4edac_row17_col18, #T_4edac_row17_col19, #T_4edac_row17_col20, #T_4edac_row17_col21, #T_4edac_row17_col22, #T_4edac_row17_col23, #T_4edac_row17_col24, #T_4edac_row17_col25, #T_4edac_row17_col26, #T_4edac_row17_col27, #T_4edac_row17_col28, #T_4edac_row17_col29, #T_4edac_row17_col30, #T_4edac_row18_col0, #T_4edac_row18_col1, #T_4edac_row18_col2, #T_4edac_row18_col3, #T_4edac_row18_col4, #T_4edac_row18_col5, #T_4edac_row18_col6, #T_4edac_row18_col7, #T_4edac_row18_col8, #T_4edac_row18_col9, #T_4edac_row18_col10, #T_4edac_row18_col11, #T_4edac_row18_col12, #T_4edac_row18_col13, #T_4edac_row18_col14, #T_4edac_row18_col15, #T_4edac_row18_col16, #T_4edac_row18_col17, #T_4edac_row18_col18, #T_4edac_row18_col19, #T_4edac_row18_col20, #T_4edac_row18_col21, #T_4edac_row18_col22, #T_4edac_row18_col23, #T_4edac_row18_col24, #T_4edac_row18_col25, #T_4edac_row18_col26, #T_4edac_row18_col27, #T_4edac_row18_col28, #T_4edac_row18_col29, #T_4edac_row18_col30, #T_4edac_row19_col0, #T_4edac_row19_col1, #T_4edac_row19_col2, #T_4edac_row19_col3, #T_4edac_row19_col4, #T_4edac_row19_col5, #T_4edac_row19_col6, #T_4edac_row19_col7, #T_4edac_row19_col8, #T_4edac_row19_col9, #T_4edac_row19_col10, #T_4edac_row19_col11, #T_4edac_row19_col12, #T_4edac_row19_col13, #T_4edac_row19_col14, #T_4edac_row19_col15, #T_4edac_row19_col16, #T_4edac_row19_col17, #T_4edac_row19_col18, #T_4edac_row19_col19, #T_4edac_row19_col20, #T_4edac_row19_col21, #T_4edac_row19_col22, #T_4edac_row19_col23, #T_4edac_row19_col24, #T_4edac_row19_col25, #T_4edac_row19_col26, #T_4edac_row19_col27, #T_4edac_row19_col28, #T_4edac_row19_col29, #T_4edac_row19_col30, #T_4edac_row20_col0, #T_4edac_row20_col1, #T_4edac_row20_col2, #T_4edac_row20_col3, #T_4edac_row20_col4, #T_4edac_row20_col5, #T_4edac_row20_col6, #T_4edac_row20_col7, #T_4edac_row20_col8, #T_4edac_row20_col9, #T_4edac_row20_col10, #T_4edac_row20_col11, #T_4edac_row20_col12, #T_4edac_row20_col13, #T_4edac_row20_col14, #T_4edac_row20_col15, #T_4edac_row20_col16, #T_4edac_row20_col17, #T_4edac_row20_col18, #T_4edac_row20_col19, #T_4edac_row20_col20, #T_4edac_row20_col21, #T_4edac_row20_col22, #T_4edac_row20_col23, #T_4edac_row20_col24, #T_4edac_row20_col25, #T_4edac_row20_col26, #T_4edac_row20_col27, #T_4edac_row20_col28, #T_4edac_row20_col29, #T_4edac_row20_col30, #T_4edac_row21_col0, #T_4edac_row21_col1, #T_4edac_row21_col2, #T_4edac_row21_col3, #T_4edac_row21_col4, #T_4edac_row21_col5, #T_4edac_row21_col6, #T_4edac_row21_col7, #T_4edac_row21_col8, #T_4edac_row21_col9, #T_4edac_row21_col10, #T_4edac_row21_col11, #T_4edac_row21_col12, #T_4edac_row21_col13, #T_4edac_row21_col14, #T_4edac_row21_col15, #T_4edac_row21_col16, #T_4edac_row21_col17, #T_4edac_row21_col18, #T_4edac_row21_col19, #T_4edac_row21_col20, #T_4edac_row21_col21, #T_4edac_row21_col22, #T_4edac_row21_col23, #T_4edac_row21_col24, #T_4edac_row21_col25, #T_4edac_row21_col26, #T_4edac_row21_col27, #T_4edac_row21_col28, #T_4edac_row21_col29, #T_4edac_row21_col30, #T_4edac_row22_col0, #T_4edac_row22_col1, #T_4edac_row22_col2, #T_4edac_row22_col3, #T_4edac_row22_col4, #T_4edac_row22_col5, #T_4edac_row22_col6, #T_4edac_row22_col7, #T_4edac_row22_col8, #T_4edac_row22_col9, #T_4edac_row22_col10, #T_4edac_row22_col11, #T_4edac_row22_col12, #T_4edac_row22_col13, #T_4edac_row22_col14, #T_4edac_row22_col15, #T_4edac_row22_col16, #T_4edac_row22_col17, #T_4edac_row22_col18, #T_4edac_row22_col19, #T_4edac_row22_col20, #T_4edac_row22_col21, #T_4edac_row22_col22, #T_4edac_row22_col23, #T_4edac_row22_col24, #T_4edac_row22_col25, #T_4edac_row22_col26, #T_4edac_row22_col27, #T_4edac_row22_col28, #T_4edac_row22_col29, #T_4edac_row22_col30, #T_4edac_row23_col0, #T_4edac_row23_col1, #T_4edac_row23_col2, #T_4edac_row23_col3, #T_4edac_row23_col4, #T_4edac_row23_col5, #T_4edac_row23_col6, #T_4edac_row23_col7, #T_4edac_row23_col8, #T_4edac_row23_col9, #T_4edac_row23_col10, #T_4edac_row23_col11, #T_4edac_row23_col12, #T_4edac_row23_col13, #T_4edac_row23_col14, #T_4edac_row23_col15, #T_4edac_row23_col16, #T_4edac_row23_col17, #T_4edac_row23_col18, #T_4edac_row23_col19, #T_4edac_row23_col20, #T_4edac_row23_col21, #T_4edac_row23_col22, #T_4edac_row23_col23, #T_4edac_row23_col24, #T_4edac_row23_col25, #T_4edac_row23_col26, #T_4edac_row23_col27, #T_4edac_row23_col28, #T_4edac_row23_col29, #T_4edac_row23_col30, #T_4edac_row24_col0, #T_4edac_row24_col1, #T_4edac_row24_col2, #T_4edac_row24_col3, #T_4edac_row24_col4, #T_4edac_row24_col5, #T_4edac_row24_col6, #T_4edac_row24_col7, #T_4edac_row24_col8, #T_4edac_row24_col9, #T_4edac_row24_col10, #T_4edac_row24_col11, #T_4edac_row24_col12, #T_4edac_row24_col13, #T_4edac_row24_col14, #T_4edac_row24_col15, #T_4edac_row24_col16, #T_4edac_row24_col17, #T_4edac_row24_col18, #T_4edac_row24_col19, #T_4edac_row24_col20, #T_4edac_row24_col21, #T_4edac_row24_col22, #T_4edac_row24_col23, #T_4edac_row24_col24, #T_4edac_row24_col25, #T_4edac_row24_col26, #T_4edac_row24_col27, #T_4edac_row24_col28, #T_4edac_row24_col29, #T_4edac_row24_col30, #T_4edac_row25_col0, #T_4edac_row25_col1, #T_4edac_row25_col2, #T_4edac_row25_col3, #T_4edac_row25_col4, #T_4edac_row25_col5, #T_4edac_row25_col6, #T_4edac_row25_col7, #T_4edac_row25_col8, #T_4edac_row25_col9, #T_4edac_row25_col10, #T_4edac_row25_col11, #T_4edac_row25_col12, #T_4edac_row25_col13, #T_4edac_row25_col14, #T_4edac_row25_col15, #T_4edac_row25_col16, #T_4edac_row25_col17, #T_4edac_row25_col18, #T_4edac_row25_col19, #T_4edac_row25_col20, #T_4edac_row25_col21, #T_4edac_row25_col22, #T_4edac_row25_col23, #T_4edac_row25_col24, #T_4edac_row25_col25, #T_4edac_row25_col26, #T_4edac_row25_col27, #T_4edac_row25_col28, #T_4edac_row25_col29, #T_4edac_row25_col30, #T_4edac_row26_col0, #T_4edac_row26_col1, #T_4edac_row26_col2, #T_4edac_row26_col3, #T_4edac_row26_col4, #T_4edac_row26_col5, #T_4edac_row26_col6, #T_4edac_row26_col7, #T_4edac_row26_col8, #T_4edac_row26_col9, #T_4edac_row26_col10, #T_4edac_row26_col11, #T_4edac_row26_col12, #T_4edac_row26_col13, #T_4edac_row26_col14, #T_4edac_row26_col15, #T_4edac_row26_col16, #T_4edac_row26_col17, #T_4edac_row26_col18, #T_4edac_row26_col19, #T_4edac_row26_col20, #T_4edac_row26_col21, #T_4edac_row26_col22, #T_4edac_row26_col23, #T_4edac_row26_col24, #T_4edac_row26_col25, #T_4edac_row26_col26, #T_4edac_row26_col27, #T_4edac_row26_col28, #T_4edac_row26_col29, #T_4edac_row26_col30, #T_4edac_row27_col0, #T_4edac_row27_col1, #T_4edac_row27_col2, #T_4edac_row27_col3, #T_4edac_row27_col4, #T_4edac_row27_col5, #T_4edac_row27_col6, #T_4edac_row27_col7, #T_4edac_row27_col8, #T_4edac_row27_col9, #T_4edac_row27_col10, #T_4edac_row27_col11, #T_4edac_row27_col12, #T_4edac_row27_col13, #T_4edac_row27_col14, #T_4edac_row27_col15, #T_4edac_row27_col16, #T_4edac_row27_col17, #T_4edac_row27_col18, #T_4edac_row27_col19, #T_4edac_row27_col20, #T_4edac_row27_col21, #T_4edac_row27_col22, #T_4edac_row27_col23, #T_4edac_row27_col24, #T_4edac_row27_col25, #T_4edac_row27_col26, #T_4edac_row27_col27, #T_4edac_row27_col28, #T_4edac_row27_col29, #T_4edac_row27_col30, #T_4edac_row28_col0, #T_4edac_row28_col1, #T_4edac_row28_col2, #T_4edac_row28_col3, #T_4edac_row28_col4, #T_4edac_row28_col5, #T_4edac_row28_col6, #T_4edac_row28_col7, #T_4edac_row28_col8, #T_4edac_row28_col9, #T_4edac_row28_col10, #T_4edac_row28_col11, #T_4edac_row28_col12, #T_4edac_row28_col13, #T_4edac_row28_col14, #T_4edac_row28_col15, #T_4edac_row28_col16, #T_4edac_row28_col17, #T_4edac_row28_col18, #T_4edac_row28_col19, #T_4edac_row28_col20, #T_4edac_row28_col21, #T_4edac_row28_col22, #T_4edac_row28_col23, #T_4edac_row28_col24, #T_4edac_row28_col25, #T_4edac_row28_col26, #T_4edac_row28_col27, #T_4edac_row28_col28, #T_4edac_row28_col29, #T_4edac_row28_col30, #T_4edac_row29_col0, #T_4edac_row29_col1, #T_4edac_row29_col2, #T_4edac_row29_col3, #T_4edac_row29_col4, #T_4edac_row29_col5, #T_4edac_row29_col6, #T_4edac_row29_col7, #T_4edac_row29_col8, #T_4edac_row29_col9, #T_4edac_row29_col10, #T_4edac_row29_col11, #T_4edac_row29_col12, #T_4edac_row29_col13, #T_4edac_row29_col14, #T_4edac_row29_col15, #T_4edac_row29_col16, #T_4edac_row29_col17, #T_4edac_row29_col18, #T_4edac_row29_col19, #T_4edac_row29_col20, #T_4edac_row29_col21, #T_4edac_row29_col22, #T_4edac_row29_col23, #T_4edac_row29_col24, #T_4edac_row29_col25, #T_4edac_row29_col26, #T_4edac_row29_col27, #T_4edac_row29_col28, #T_4edac_row29_col29, #T_4edac_row29_col30, #T_4edac_row30_col0, #T_4edac_row30_col1, #T_4edac_row30_col2, #T_4edac_row30_col3, #T_4edac_row30_col4, #T_4edac_row30_col5, #T_4edac_row30_col6, #T_4edac_row30_col7, #T_4edac_row30_col8, #T_4edac_row30_col9, #T_4edac_row30_col10, #T_4edac_row30_col11, #T_4edac_row30_col12, #T_4edac_row30_col13, #T_4edac_row30_col14, #T_4edac_row30_col15, #T_4edac_row30_col16, #T_4edac_row30_col17, #T_4edac_row30_col18, #T_4edac_row30_col19, #T_4edac_row30_col20, #T_4edac_row30_col21, #T_4edac_row30_col22, #T_4edac_row30_col23, #T_4edac_row30_col24, #T_4edac_row30_col25, #T_4edac_row30_col26, #T_4edac_row30_col27, #T_4edac_row30_col28, #T_4edac_row30_col29, #T_4edac_row30_col30 {
  font-size: 6pt;
  font-family: Arial;
}
</style>
<table id="T_4edac">
  <thead>
    <tr>
      <th class="blank level0" >&nbsp;</th>
      <th id="T_4edac_level0_col0" class="col_heading level0 col0" >radius_mean</th>
      <th id="T_4edac_level0_col1" class="col_heading level0 col1" >texture_mean</th>
      <th id="T_4edac_level0_col2" class="col_heading level0 col2" >perimeter_mean</th>
      <th id="T_4edac_level0_col3" class="col_heading level0 col3" >area_mean</th>
      <th id="T_4edac_level0_col4" class="col_heading level0 col4" >smoothness_mean</th>
      <th id="T_4edac_level0_col5" class="col_heading level0 col5" >compactness_mean</th>
      <th id="T_4edac_level0_col6" class="col_heading level0 col6" >concavity_mean</th>
      <th id="T_4edac_level0_col7" class="col_heading level0 col7" >concave points_mean</th>
      <th id="T_4edac_level0_col8" class="col_heading level0 col8" >symmetry_mean</th>
      <th id="T_4edac_level0_col9" class="col_heading level0 col9" >fractal_dimension_mean</th>
      <th id="T_4edac_level0_col10" class="col_heading level0 col10" >radius_se</th>
      <th id="T_4edac_level0_col11" class="col_heading level0 col11" >texture_se</th>
      <th id="T_4edac_level0_col12" class="col_heading level0 col12" >perimeter_se</th>
      <th id="T_4edac_level0_col13" class="col_heading level0 col13" >area_se</th>
      <th id="T_4edac_level0_col14" class="col_heading level0 col14" >smoothness_se</th>
      <th id="T_4edac_level0_col15" class="col_heading level0 col15" >compactness_se</th>
      <th id="T_4edac_level0_col16" class="col_heading level0 col16" >concavity_se</th>
      <th id="T_4edac_level0_col17" class="col_heading level0 col17" >concave points_se</th>
      <th id="T_4edac_level0_col18" class="col_heading level0 col18" >symmetry_se</th>
      <th id="T_4edac_level0_col19" class="col_heading level0 col19" >fractal_dimension_se</th>
      <th id="T_4edac_level0_col20" class="col_heading level0 col20" >radius_worst</th>
      <th id="T_4edac_level0_col21" class="col_heading level0 col21" >texture_worst</th>
      <th id="T_4edac_level0_col22" class="col_heading level0 col22" >perimeter_worst</th>
      <th id="T_4edac_level0_col23" class="col_heading level0 col23" >area_worst</th>
      <th id="T_4edac_level0_col24" class="col_heading level0 col24" >smoothness_worst</th>
      <th id="T_4edac_level0_col25" class="col_heading level0 col25" >compactness_worst</th>
      <th id="T_4edac_level0_col26" class="col_heading level0 col26" >concavity_worst</th>
      <th id="T_4edac_level0_col27" class="col_heading level0 col27" >concave points_worst</th>
      <th id="T_4edac_level0_col28" class="col_heading level0 col28" >symmetry_worst</th>
      <th id="T_4edac_level0_col29" class="col_heading level0 col29" >fractal_dimension_worst</th>
      <th id="T_4edac_level0_col30" class="col_heading level0 col30" >diagnosis</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th id="T_4edac_level0_row0" class="row_heading level0 row0" >radius_mean</th>
      <td id="T_4edac_row0_col0" class="data row0 col0" >1.00</td>
      <td id="T_4edac_row0_col1" class="data row0 col1" >0.39</td>
      <td id="T_4edac_row0_col2" class="data row0 col2" >1.00</td>
      <td id="T_4edac_row0_col3" class="data row0 col3" >0.99</td>
      <td id="T_4edac_row0_col4" class="data row0 col4" >0.16</td>
      <td id="T_4edac_row0_col5" class="data row0 col5" >0.53</td>
      <td id="T_4edac_row0_col6" class="data row0 col6" >0.65</td>
      <td id="T_4edac_row0_col7" class="data row0 col7" >0.81</td>
      <td id="T_4edac_row0_col8" class="data row0 col8" >0.17</td>
      <td id="T_4edac_row0_col9" class="data row0 col9" >-0.31</td>
      <td id="T_4edac_row0_col10" class="data row0 col10" >0.70</td>
      <td id="T_4edac_row0_col11" class="data row0 col11" >-0.08</td>
      <td id="T_4edac_row0_col12" class="data row0 col12" >0.70</td>
      <td id="T_4edac_row0_col13" class="data row0 col13" >0.76</td>
      <td id="T_4edac_row0_col14" class="data row0 col14" >-0.20</td>
      <td id="T_4edac_row0_col15" class="data row0 col15" >0.21</td>
      <td id="T_4edac_row0_col16" class="data row0 col16" >0.15</td>
      <td id="T_4edac_row0_col17" class="data row0 col17" >0.32</td>
      <td id="T_4edac_row0_col18" class="data row0 col18" >-0.12</td>
      <td id="T_4edac_row0_col19" class="data row0 col19" >-0.06</td>
      <td id="T_4edac_row0_col20" class="data row0 col20" >0.97</td>
      <td id="T_4edac_row0_col21" class="data row0 col21" >0.36</td>
      <td id="T_4edac_row0_col22" class="data row0 col22" >0.97</td>
      <td id="T_4edac_row0_col23" class="data row0 col23" >0.94</td>
      <td id="T_4edac_row0_col24" class="data row0 col24" >0.15</td>
      <td id="T_4edac_row0_col25" class="data row0 col25" >0.45</td>
      <td id="T_4edac_row0_col26" class="data row0 col26" >0.53</td>
      <td id="T_4edac_row0_col27" class="data row0 col27" >0.74</td>
      <td id="T_4edac_row0_col28" class="data row0 col28" >0.20</td>
      <td id="T_4edac_row0_col29" class="data row0 col29" >0.04</td>
      <td id="T_4edac_row0_col30" class="data row0 col30" >0.74</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row1" class="row_heading level0 row1" >texture_mean</th>
      <td id="T_4edac_row1_col0" class="data row1 col0" >0.39</td>
      <td id="T_4edac_row1_col1" class="data row1 col1" >1.00</td>
      <td id="T_4edac_row1_col2" class="data row1 col2" >0.39</td>
      <td id="T_4edac_row1_col3" class="data row1 col3" >0.38</td>
      <td id="T_4edac_row1_col4" class="data row1 col4" >-0.05</td>
      <td id="T_4edac_row1_col5" class="data row1 col5" >0.26</td>
      <td id="T_4edac_row1_col6" class="data row1 col6" >0.34</td>
      <td id="T_4edac_row1_col7" class="data row1 col7" >0.35</td>
      <td id="T_4edac_row1_col8" class="data row1 col8" >0.08</td>
      <td id="T_4edac_row1_col9" class="data row1 col9" >-0.11</td>
      <td id="T_4edac_row1_col10" class="data row1 col10" >0.33</td>
      <td id="T_4edac_row1_col11" class="data row1 col11" >0.36</td>
      <td id="T_4edac_row1_col12" class="data row1 col12" >0.33</td>
      <td id="T_4edac_row1_col13" class="data row1 col13" >0.32</td>
      <td id="T_4edac_row1_col14" class="data row1 col14" >-0.03</td>
      <td id="T_4edac_row1_col15" class="data row1 col15" >0.19</td>
      <td id="T_4edac_row1_col16" class="data row1 col16" >0.13</td>
      <td id="T_4edac_row1_col17" class="data row1 col17" >0.18</td>
      <td id="T_4edac_row1_col18" class="data row1 col18" >-0.00</td>
      <td id="T_4edac_row1_col19" class="data row1 col19" >0.03</td>
      <td id="T_4edac_row1_col20" class="data row1 col20" >0.41</td>
      <td id="T_4edac_row1_col21" class="data row1 col21" >0.92</td>
      <td id="T_4edac_row1_col22" class="data row1 col22" >0.42</td>
      <td id="T_4edac_row1_col23" class="data row1 col23" >0.40</td>
      <td id="T_4edac_row1_col24" class="data row1 col24" >0.05</td>
      <td id="T_4edac_row1_col25" class="data row1 col25" >0.32</td>
      <td id="T_4edac_row1_col26" class="data row1 col26" >0.33</td>
      <td id="T_4edac_row1_col27" class="data row1 col27" >0.35</td>
      <td id="T_4edac_row1_col28" class="data row1 col28" >0.12</td>
      <td id="T_4edac_row1_col29" class="data row1 col29" >0.12</td>
      <td id="T_4edac_row1_col30" class="data row1 col30" >0.46</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row2" class="row_heading level0 row2" >perimeter_mean</th>
      <td id="T_4edac_row2_col0" class="data row2 col0" >1.00</td>
      <td id="T_4edac_row2_col1" class="data row2 col1" >0.39</td>
      <td id="T_4edac_row2_col2" class="data row2 col2" >1.00</td>
      <td id="T_4edac_row2_col3" class="data row2 col3" >0.99</td>
      <td id="T_4edac_row2_col4" class="data row2 col4" >0.19</td>
      <td id="T_4edac_row2_col5" class="data row2 col5" >0.58</td>
      <td id="T_4edac_row2_col6" class="data row2 col6" >0.69</td>
      <td id="T_4edac_row2_col7" class="data row2 col7" >0.84</td>
      <td id="T_4edac_row2_col8" class="data row2 col8" >0.20</td>
      <td id="T_4edac_row2_col9" class="data row2 col9" >-0.26</td>
      <td id="T_4edac_row2_col10" class="data row2 col10" >0.71</td>
      <td id="T_4edac_row2_col11" class="data row2 col11" >-0.07</td>
      <td id="T_4edac_row2_col12" class="data row2 col12" >0.72</td>
      <td id="T_4edac_row2_col13" class="data row2 col13" >0.77</td>
      <td id="T_4edac_row2_col14" class="data row2 col14" >-0.18</td>
      <td id="T_4edac_row2_col15" class="data row2 col15" >0.25</td>
      <td id="T_4edac_row2_col16" class="data row2 col16" >0.18</td>
      <td id="T_4edac_row2_col17" class="data row2 col17" >0.35</td>
      <td id="T_4edac_row2_col18" class="data row2 col18" >-0.09</td>
      <td id="T_4edac_row2_col19" class="data row2 col19" >-0.02</td>
      <td id="T_4edac_row2_col20" class="data row2 col20" >0.97</td>
      <td id="T_4edac_row2_col21" class="data row2 col21" >0.37</td>
      <td id="T_4edac_row2_col22" class="data row2 col22" >0.97</td>
      <td id="T_4edac_row2_col23" class="data row2 col23" >0.94</td>
      <td id="T_4edac_row2_col24" class="data row2 col24" >0.18</td>
      <td id="T_4edac_row2_col25" class="data row2 col25" >0.49</td>
      <td id="T_4edac_row2_col26" class="data row2 col26" >0.56</td>
      <td id="T_4edac_row2_col27" class="data row2 col27" >0.76</td>
      <td id="T_4edac_row2_col28" class="data row2 col28" >0.22</td>
      <td id="T_4edac_row2_col29" class="data row2 col29" >0.09</td>
      <td id="T_4edac_row2_col30" class="data row2 col30" >0.75</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row3" class="row_heading level0 row3" >area_mean</th>
      <td id="T_4edac_row3_col0" class="data row3 col0" >0.99</td>
      <td id="T_4edac_row3_col1" class="data row3 col1" >0.38</td>
      <td id="T_4edac_row3_col2" class="data row3 col2" >0.99</td>
      <td id="T_4edac_row3_col3" class="data row3 col3" >1.00</td>
      <td id="T_4edac_row3_col4" class="data row3 col4" >0.17</td>
      <td id="T_4edac_row3_col5" class="data row3 col5" >0.53</td>
      <td id="T_4edac_row3_col6" class="data row3 col6" >0.67</td>
      <td id="T_4edac_row3_col7" class="data row3 col7" >0.82</td>
      <td id="T_4edac_row3_col8" class="data row3 col8" >0.18</td>
      <td id="T_4edac_row3_col9" class="data row3 col9" >-0.27</td>
      <td id="T_4edac_row3_col10" class="data row3 col10" >0.75</td>
      <td id="T_4edac_row3_col11" class="data row3 col11" >-0.06</td>
      <td id="T_4edac_row3_col12" class="data row3 col12" >0.74</td>
      <td id="T_4edac_row3_col13" class="data row3 col13" >0.82</td>
      <td id="T_4edac_row3_col14" class="data row3 col14" >-0.15</td>
      <td id="T_4edac_row3_col15" class="data row3 col15" >0.22</td>
      <td id="T_4edac_row3_col16" class="data row3 col16" >0.17</td>
      <td id="T_4edac_row3_col17" class="data row3 col17" >0.33</td>
      <td id="T_4edac_row3_col18" class="data row3 col18" >-0.09</td>
      <td id="T_4edac_row3_col19" class="data row3 col19" >-0.03</td>
      <td id="T_4edac_row3_col20" class="data row3 col20" >0.97</td>
      <td id="T_4edac_row3_col21" class="data row3 col21" >0.35</td>
      <td id="T_4edac_row3_col22" class="data row3 col22" >0.97</td>
      <td id="T_4edac_row3_col23" class="data row3 col23" >0.97</td>
      <td id="T_4edac_row3_col24" class="data row3 col24" >0.15</td>
      <td id="T_4edac_row3_col25" class="data row3 col25" >0.43</td>
      <td id="T_4edac_row3_col26" class="data row3 col26" >0.52</td>
      <td id="T_4edac_row3_col27" class="data row3 col27" >0.73</td>
      <td id="T_4edac_row3_col28" class="data row3 col28" >0.18</td>
      <td id="T_4edac_row3_col29" class="data row3 col29" >0.04</td>
      <td id="T_4edac_row3_col30" class="data row3 col30" >0.73</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row4" class="row_heading level0 row4" >smoothness_mean</th>
      <td id="T_4edac_row4_col0" class="data row4 col0" >0.16</td>
      <td id="T_4edac_row4_col1" class="data row4 col1" >-0.05</td>
      <td id="T_4edac_row4_col2" class="data row4 col2" >0.19</td>
      <td id="T_4edac_row4_col3" class="data row4 col3" >0.17</td>
      <td id="T_4edac_row4_col4" class="data row4 col4" >1.00</td>
      <td id="T_4edac_row4_col5" class="data row4 col5" >0.64</td>
      <td id="T_4edac_row4_col6" class="data row4 col6" >0.49</td>
      <td id="T_4edac_row4_col7" class="data row4 col7" >0.53</td>
      <td id="T_4edac_row4_col8" class="data row4 col8" >0.58</td>
      <td id="T_4edac_row4_col9" class="data row4 col9" >0.58</td>
      <td id="T_4edac_row4_col10" class="data row4 col10" >0.30</td>
      <td id="T_4edac_row4_col11" class="data row4 col11" >0.09</td>
      <td id="T_4edac_row4_col12" class="data row4 col12" >0.31</td>
      <td id="T_4edac_row4_col13" class="data row4 col13" >0.24</td>
      <td id="T_4edac_row4_col14" class="data row4 col14" >0.32</td>
      <td id="T_4edac_row4_col15" class="data row4 col15" >0.31</td>
      <td id="T_4edac_row4_col16" class="data row4 col16" >0.23</td>
      <td id="T_4edac_row4_col17" class="data row4 col17" >0.36</td>
      <td id="T_4edac_row4_col18" class="data row4 col18" >0.22</td>
      <td id="T_4edac_row4_col19" class="data row4 col19" >0.25</td>
      <td id="T_4edac_row4_col20" class="data row4 col20" >0.20</td>
      <td id="T_4edac_row4_col21" class="data row4 col21" >0.02</td>
      <td id="T_4edac_row4_col22" class="data row4 col22" >0.23</td>
      <td id="T_4edac_row4_col23" class="data row4 col23" >0.20</td>
      <td id="T_4edac_row4_col24" class="data row4 col24" >0.79</td>
      <td id="T_4edac_row4_col25" class="data row4 col25" >0.45</td>
      <td id="T_4edac_row4_col26" class="data row4 col26" >0.41</td>
      <td id="T_4edac_row4_col27" class="data row4 col27" >0.47</td>
      <td id="T_4edac_row4_col28" class="data row4 col28" >0.39</td>
      <td id="T_4edac_row4_col29" class="data row4 col29" >0.47</td>
      <td id="T_4edac_row4_col30" class="data row4 col30" >0.33</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row5" class="row_heading level0 row5" >compactness_mean</th>
      <td id="T_4edac_row5_col0" class="data row5 col0" >0.53</td>
      <td id="T_4edac_row5_col1" class="data row5 col1" >0.26</td>
      <td id="T_4edac_row5_col2" class="data row5 col2" >0.58</td>
      <td id="T_4edac_row5_col3" class="data row5 col3" >0.53</td>
      <td id="T_4edac_row5_col4" class="data row5 col4" >0.64</td>
      <td id="T_4edac_row5_col5" class="data row5 col5" >1.00</td>
      <td id="T_4edac_row5_col6" class="data row5 col6" >0.88</td>
      <td id="T_4edac_row5_col7" class="data row5 col7" >0.83</td>
      <td id="T_4edac_row5_col8" class="data row5 col8" >0.59</td>
      <td id="T_4edac_row5_col9" class="data row5 col9" >0.55</td>
      <td id="T_4edac_row5_col10" class="data row5 col10" >0.53</td>
      <td id="T_4edac_row5_col11" class="data row5 col11" >0.06</td>
      <td id="T_4edac_row5_col12" class="data row5 col12" >0.60</td>
      <td id="T_4edac_row5_col13" class="data row5 col13" >0.50</td>
      <td id="T_4edac_row5_col14" class="data row5 col14" >0.14</td>
      <td id="T_4edac_row5_col15" class="data row5 col15" >0.73</td>
      <td id="T_4edac_row5_col16" class="data row5 col16" >0.54</td>
      <td id="T_4edac_row5_col17" class="data row5 col17" >0.62</td>
      <td id="T_4edac_row5_col18" class="data row5 col18" >0.24</td>
      <td id="T_4edac_row5_col19" class="data row5 col19" >0.47</td>
      <td id="T_4edac_row5_col20" class="data row5 col20" >0.55</td>
      <td id="T_4edac_row5_col21" class="data row5 col21" >0.28</td>
      <td id="T_4edac_row5_col22" class="data row5 col22" >0.61</td>
      <td id="T_4edac_row5_col23" class="data row5 col23" >0.53</td>
      <td id="T_4edac_row5_col24" class="data row5 col24" >0.54</td>
      <td id="T_4edac_row5_col25" class="data row5 col25" >0.88</td>
      <td id="T_4edac_row5_col26" class="data row5 col26" >0.81</td>
      <td id="T_4edac_row5_col27" class="data row5 col27" >0.81</td>
      <td id="T_4edac_row5_col28" class="data row5 col28" >0.50</td>
      <td id="T_4edac_row5_col29" class="data row5 col29" >0.69</td>
      <td id="T_4edac_row5_col30" class="data row5 col30" >0.59</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row6" class="row_heading level0 row6" >concavity_mean</th>
      <td id="T_4edac_row6_col0" class="data row6 col0" >0.65</td>
      <td id="T_4edac_row6_col1" class="data row6 col1" >0.34</td>
      <td id="T_4edac_row6_col2" class="data row6 col2" >0.69</td>
      <td id="T_4edac_row6_col3" class="data row6 col3" >0.67</td>
      <td id="T_4edac_row6_col4" class="data row6 col4" >0.49</td>
      <td id="T_4edac_row6_col5" class="data row6 col5" >0.88</td>
      <td id="T_4edac_row6_col6" class="data row6 col6" >1.00</td>
      <td id="T_4edac_row6_col7" class="data row6 col7" >0.91</td>
      <td id="T_4edac_row6_col8" class="data row6 col8" >0.52</td>
      <td id="T_4edac_row6_col9" class="data row6 col9" >0.36</td>
      <td id="T_4edac_row6_col10" class="data row6 col10" >0.66</td>
      <td id="T_4edac_row6_col11" class="data row6 col11" >0.13</td>
      <td id="T_4edac_row6_col12" class="data row6 col12" >0.70</td>
      <td id="T_4edac_row6_col13" class="data row6 col13" >0.64</td>
      <td id="T_4edac_row6_col14" class="data row6 col14" >0.14</td>
      <td id="T_4edac_row6_col15" class="data row6 col15" >0.68</td>
      <td id="T_4edac_row6_col16" class="data row6 col16" >0.69</td>
      <td id="T_4edac_row6_col17" class="data row6 col17" >0.70</td>
      <td id="T_4edac_row6_col18" class="data row6 col18" >0.22</td>
      <td id="T_4edac_row6_col19" class="data row6 col19" >0.47</td>
      <td id="T_4edac_row6_col20" class="data row6 col20" >0.67</td>
      <td id="T_4edac_row6_col21" class="data row6 col21" >0.32</td>
      <td id="T_4edac_row6_col22" class="data row6 col22" >0.71</td>
      <td id="T_4edac_row6_col23" class="data row6 col23" >0.66</td>
      <td id="T_4edac_row6_col24" class="data row6 col24" >0.42</td>
      <td id="T_4edac_row6_col25" class="data row6 col25" >0.75</td>
      <td id="T_4edac_row6_col26" class="data row6 col26" >0.89</td>
      <td id="T_4edac_row6_col27" class="data row6 col27" >0.86</td>
      <td id="T_4edac_row6_col28" class="data row6 col28" >0.40</td>
      <td id="T_4edac_row6_col29" class="data row6 col29" >0.53</td>
      <td id="T_4edac_row6_col30" class="data row6 col30" >0.67</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row7" class="row_heading level0 row7" >concave points_mean</th>
      <td id="T_4edac_row7_col0" class="data row7 col0" >0.81</td>
      <td id="T_4edac_row7_col1" class="data row7 col1" >0.35</td>
      <td id="T_4edac_row7_col2" class="data row7 col2" >0.84</td>
      <td id="T_4edac_row7_col3" class="data row7 col3" >0.82</td>
      <td id="T_4edac_row7_col4" class="data row7 col4" >0.53</td>
      <td id="T_4edac_row7_col5" class="data row7 col5" >0.83</td>
      <td id="T_4edac_row7_col6" class="data row7 col6" >0.91</td>
      <td id="T_4edac_row7_col7" class="data row7 col7" >1.00</td>
      <td id="T_4edac_row7_col8" class="data row7 col8" >0.47</td>
      <td id="T_4edac_row7_col9" class="data row7 col9" >0.18</td>
      <td id="T_4edac_row7_col10" class="data row7 col10" >0.73</td>
      <td id="T_4edac_row7_col11" class="data row7 col11" >0.07</td>
      <td id="T_4edac_row7_col12" class="data row7 col12" >0.75</td>
      <td id="T_4edac_row7_col13" class="data row7 col13" >0.72</td>
      <td id="T_4edac_row7_col14" class="data row7 col14" >0.06</td>
      <td id="T_4edac_row7_col15" class="data row7 col15" >0.49</td>
      <td id="T_4edac_row7_col16" class="data row7 col16" >0.41</td>
      <td id="T_4edac_row7_col17" class="data row7 col17" >0.59</td>
      <td id="T_4edac_row7_col18" class="data row7 col18" >0.11</td>
      <td id="T_4edac_row7_col19" class="data row7 col19" >0.25</td>
      <td id="T_4edac_row7_col20" class="data row7 col20" >0.82</td>
      <td id="T_4edac_row7_col21" class="data row7 col21" >0.34</td>
      <td id="T_4edac_row7_col22" class="data row7 col22" >0.85</td>
      <td id="T_4edac_row7_col23" class="data row7 col23" >0.81</td>
      <td id="T_4edac_row7_col24" class="data row7 col24" >0.45</td>
      <td id="T_4edac_row7_col25" class="data row7 col25" >0.68</td>
      <td id="T_4edac_row7_col26" class="data row7 col26" >0.76</td>
      <td id="T_4edac_row7_col27" class="data row7 col27" >0.91</td>
      <td id="T_4edac_row7_col28" class="data row7 col28" >0.37</td>
      <td id="T_4edac_row7_col29" class="data row7 col29" >0.40</td>
      <td id="T_4edac_row7_col30" class="data row7 col30" >0.77</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row8" class="row_heading level0 row8" >symmetry_mean</th>
      <td id="T_4edac_row8_col0" class="data row8 col0" >0.17</td>
      <td id="T_4edac_row8_col1" class="data row8 col1" >0.08</td>
      <td id="T_4edac_row8_col2" class="data row8 col2" >0.20</td>
      <td id="T_4edac_row8_col3" class="data row8 col3" >0.18</td>
      <td id="T_4edac_row8_col4" class="data row8 col4" >0.58</td>
      <td id="T_4edac_row8_col5" class="data row8 col5" >0.59</td>
      <td id="T_4edac_row8_col6" class="data row8 col6" >0.52</td>
      <td id="T_4edac_row8_col7" class="data row8 col7" >0.47</td>
      <td id="T_4edac_row8_col8" class="data row8 col8" >1.00</td>
      <td id="T_4edac_row8_col9" class="data row8 col9" >0.45</td>
      <td id="T_4edac_row8_col10" class="data row8 col10" >0.32</td>
      <td id="T_4edac_row8_col11" class="data row8 col11" >0.12</td>
      <td id="T_4edac_row8_col12" class="data row8 col12" >0.33</td>
      <td id="T_4edac_row8_col13" class="data row8 col13" >0.26</td>
      <td id="T_4edac_row8_col14" class="data row8 col14" >0.17</td>
      <td id="T_4edac_row8_col15" class="data row8 col15" >0.40</td>
      <td id="T_4edac_row8_col16" class="data row8 col16" >0.34</td>
      <td id="T_4edac_row8_col17" class="data row8 col17" >0.36</td>
      <td id="T_4edac_row8_col18" class="data row8 col18" >0.45</td>
      <td id="T_4edac_row8_col19" class="data row8 col19" >0.31</td>
      <td id="T_4edac_row8_col20" class="data row8 col20" >0.20</td>
      <td id="T_4edac_row8_col21" class="data row8 col21" >0.10</td>
      <td id="T_4edac_row8_col22" class="data row8 col22" >0.23</td>
      <td id="T_4edac_row8_col23" class="data row8 col23" >0.20</td>
      <td id="T_4edac_row8_col24" class="data row8 col24" >0.42</td>
      <td id="T_4edac_row8_col25" class="data row8 col25" >0.48</td>
      <td id="T_4edac_row8_col26" class="data row8 col26" >0.46</td>
      <td id="T_4edac_row8_col27" class="data row8 col27" >0.43</td>
      <td id="T_4edac_row8_col28" class="data row8 col28" >0.69</td>
      <td id="T_4edac_row8_col29" class="data row8 col29" >0.44</td>
      <td id="T_4edac_row8_col30" class="data row8 col30" >0.33</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row9" class="row_heading level0 row9" >fractal_dimension_mean</th>
      <td id="T_4edac_row9_col0" class="data row9 col0" >-0.31</td>
      <td id="T_4edac_row9_col1" class="data row9 col1" >-0.11</td>
      <td id="T_4edac_row9_col2" class="data row9 col2" >-0.26</td>
      <td id="T_4edac_row9_col3" class="data row9 col3" >-0.27</td>
      <td id="T_4edac_row9_col4" class="data row9 col4" >0.58</td>
      <td id="T_4edac_row9_col5" class="data row9 col5" >0.55</td>
      <td id="T_4edac_row9_col6" class="data row9 col6" >0.36</td>
      <td id="T_4edac_row9_col7" class="data row9 col7" >0.18</td>
      <td id="T_4edac_row9_col8" class="data row9 col8" >0.45</td>
      <td id="T_4edac_row9_col9" class="data row9 col9" >1.00</td>
      <td id="T_4edac_row9_col10" class="data row9 col10" >0.00</td>
      <td id="T_4edac_row9_col11" class="data row9 col11" >0.19</td>
      <td id="T_4edac_row9_col12" class="data row9 col12" >0.06</td>
      <td id="T_4edac_row9_col13" class="data row9 col13" >-0.09</td>
      <td id="T_4edac_row9_col14" class="data row9 col14" >0.37</td>
      <td id="T_4edac_row9_col15" class="data row9 col15" >0.55</td>
      <td id="T_4edac_row9_col16" class="data row9 col16" >0.48</td>
      <td id="T_4edac_row9_col17" class="data row9 col17" >0.40</td>
      <td id="T_4edac_row9_col18" class="data row9 col18" >0.35</td>
      <td id="T_4edac_row9_col19" class="data row9 col19" >0.68</td>
      <td id="T_4edac_row9_col20" class="data row9 col20" >-0.25</td>
      <td id="T_4edac_row9_col21" class="data row9 col21" >-0.08</td>
      <td id="T_4edac_row9_col22" class="data row9 col22" >-0.20</td>
      <td id="T_4edac_row9_col23" class="data row9 col23" >-0.22</td>
      <td id="T_4edac_row9_col24" class="data row9 col24" >0.45</td>
      <td id="T_4edac_row9_col25" class="data row9 col25" >0.43</td>
      <td id="T_4edac_row9_col26" class="data row9 col26" >0.35</td>
      <td id="T_4edac_row9_col27" class="data row9 col27" >0.19</td>
      <td id="T_4edac_row9_col28" class="data row9 col28" >0.28</td>
      <td id="T_4edac_row9_col29" class="data row9 col29" >0.75</td>
      <td id="T_4edac_row9_col30" class="data row9 col30" >-0.05</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row10" class="row_heading level0 row10" >radius_se</th>
      <td id="T_4edac_row10_col0" class="data row10 col0" >0.70</td>
      <td id="T_4edac_row10_col1" class="data row10 col1" >0.33</td>
      <td id="T_4edac_row10_col2" class="data row10 col2" >0.71</td>
      <td id="T_4edac_row10_col3" class="data row10 col3" >0.75</td>
      <td id="T_4edac_row10_col4" class="data row10 col4" >0.30</td>
      <td id="T_4edac_row10_col5" class="data row10 col5" >0.53</td>
      <td id="T_4edac_row10_col6" class="data row10 col6" >0.66</td>
      <td id="T_4edac_row10_col7" class="data row10 col7" >0.73</td>
      <td id="T_4edac_row10_col8" class="data row10 col8" >0.32</td>
      <td id="T_4edac_row10_col9" class="data row10 col9" >0.00</td>
      <td id="T_4edac_row10_col10" class="data row10 col10" >1.00</td>
      <td id="T_4edac_row10_col11" class="data row10 col11" >0.22</td>
      <td id="T_4edac_row10_col12" class="data row10 col12" >0.97</td>
      <td id="T_4edac_row10_col13" class="data row10 col13" >0.95</td>
      <td id="T_4edac_row10_col14" class="data row10 col14" >0.17</td>
      <td id="T_4edac_row10_col15" class="data row10 col15" >0.39</td>
      <td id="T_4edac_row10_col16" class="data row10 col16" >0.34</td>
      <td id="T_4edac_row10_col17" class="data row10 col17" >0.55</td>
      <td id="T_4edac_row10_col18" class="data row10 col18" >0.20</td>
      <td id="T_4edac_row10_col19" class="data row10 col19" >0.22</td>
      <td id="T_4edac_row10_col20" class="data row10 col20" >0.75</td>
      <td id="T_4edac_row10_col21" class="data row10 col21" >0.25</td>
      <td id="T_4edac_row10_col22" class="data row10 col22" >0.75</td>
      <td id="T_4edac_row10_col23" class="data row10 col23" >0.78</td>
      <td id="T_4edac_row10_col24" class="data row10 col24" >0.17</td>
      <td id="T_4edac_row10_col25" class="data row10 col25" >0.34</td>
      <td id="T_4edac_row10_col26" class="data row10 col26" >0.44</td>
      <td id="T_4edac_row10_col27" class="data row10 col27" >0.58</td>
      <td id="T_4edac_row10_col28" class="data row10 col28" >0.13</td>
      <td id="T_4edac_row10_col29" class="data row10 col29" >0.08</td>
      <td id="T_4edac_row10_col30" class="data row10 col30" >0.61</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row11" class="row_heading level0 row11" >texture_se</th>
      <td id="T_4edac_row11_col0" class="data row11 col0" >-0.08</td>
      <td id="T_4edac_row11_col1" class="data row11 col1" >0.36</td>
      <td id="T_4edac_row11_col2" class="data row11 col2" >-0.07</td>
      <td id="T_4edac_row11_col3" class="data row11 col3" >-0.06</td>
      <td id="T_4edac_row11_col4" class="data row11 col4" >0.09</td>
      <td id="T_4edac_row11_col5" class="data row11 col5" >0.06</td>
      <td id="T_4edac_row11_col6" class="data row11 col6" >0.13</td>
      <td id="T_4edac_row11_col7" class="data row11 col7" >0.07</td>
      <td id="T_4edac_row11_col8" class="data row11 col8" >0.12</td>
      <td id="T_4edac_row11_col9" class="data row11 col9" >0.19</td>
      <td id="T_4edac_row11_col10" class="data row11 col10" >0.22</td>
      <td id="T_4edac_row11_col11" class="data row11 col11" >1.00</td>
      <td id="T_4edac_row11_col12" class="data row11 col12" >0.22</td>
      <td id="T_4edac_row11_col13" class="data row11 col13" >0.12</td>
      <td id="T_4edac_row11_col14" class="data row11 col14" >0.38</td>
      <td id="T_4edac_row11_col15" class="data row11 col15" >0.26</td>
      <td id="T_4edac_row11_col16" class="data row11 col16" >0.25</td>
      <td id="T_4edac_row11_col17" class="data row11 col17" >0.30</td>
      <td id="T_4edac_row11_col18" class="data row11 col18" >0.40</td>
      <td id="T_4edac_row11_col19" class="data row11 col19" >0.30</td>
      <td id="T_4edac_row11_col20" class="data row11 col20" >-0.10</td>
      <td id="T_4edac_row11_col21" class="data row11 col21" >0.38</td>
      <td id="T_4edac_row11_col22" class="data row11 col22" >-0.09</td>
      <td id="T_4edac_row11_col23" class="data row11 col23" >-0.07</td>
      <td id="T_4edac_row11_col24" class="data row11 col24" >-0.07</td>
      <td id="T_4edac_row11_col25" class="data row11 col25" >-0.07</td>
      <td id="T_4edac_row11_col26" class="data row11 col26" >-0.01</td>
      <td id="T_4edac_row11_col27" class="data row11 col27" >-0.05</td>
      <td id="T_4edac_row11_col28" class="data row11 col28" >-0.14</td>
      <td id="T_4edac_row11_col29" class="data row11 col29" >-0.04</td>
      <td id="T_4edac_row11_col30" class="data row11 col30" >0.01</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row12" class="row_heading level0 row12" >perimeter_se</th>
      <td id="T_4edac_row12_col0" class="data row12 col0" >0.70</td>
      <td id="T_4edac_row12_col1" class="data row12 col1" >0.33</td>
      <td id="T_4edac_row12_col2" class="data row12 col2" >0.72</td>
      <td id="T_4edac_row12_col3" class="data row12 col3" >0.74</td>
      <td id="T_4edac_row12_col4" class="data row12 col4" >0.31</td>
      <td id="T_4edac_row12_col5" class="data row12 col5" >0.60</td>
      <td id="T_4edac_row12_col6" class="data row12 col6" >0.70</td>
      <td id="T_4edac_row12_col7" class="data row12 col7" >0.75</td>
      <td id="T_4edac_row12_col8" class="data row12 col8" >0.33</td>
      <td id="T_4edac_row12_col9" class="data row12 col9" >0.06</td>
      <td id="T_4edac_row12_col10" class="data row12 col10" >0.97</td>
      <td id="T_4edac_row12_col11" class="data row12 col11" >0.22</td>
      <td id="T_4edac_row12_col12" class="data row12 col12" >1.00</td>
      <td id="T_4edac_row12_col13" class="data row12 col13" >0.93</td>
      <td id="T_4edac_row12_col14" class="data row12 col14" >0.16</td>
      <td id="T_4edac_row12_col15" class="data row12 col15" >0.46</td>
      <td id="T_4edac_row12_col16" class="data row12 col16" >0.37</td>
      <td id="T_4edac_row12_col17" class="data row12 col17" >0.59</td>
      <td id="T_4edac_row12_col18" class="data row12 col18" >0.22</td>
      <td id="T_4edac_row12_col19" class="data row12 col19" >0.25</td>
      <td id="T_4edac_row12_col20" class="data row12 col20" >0.73</td>
      <td id="T_4edac_row12_col21" class="data row12 col21" >0.26</td>
      <td id="T_4edac_row12_col22" class="data row12 col22" >0.76</td>
      <td id="T_4edac_row12_col23" class="data row12 col23" >0.77</td>
      <td id="T_4edac_row12_col24" class="data row12 col24" >0.18</td>
      <td id="T_4edac_row12_col25" class="data row12 col25" >0.42</td>
      <td id="T_4edac_row12_col26" class="data row12 col26" >0.49</td>
      <td id="T_4edac_row12_col27" class="data row12 col27" >0.62</td>
      <td id="T_4edac_row12_col28" class="data row12 col28" >0.14</td>
      <td id="T_4edac_row12_col29" class="data row12 col29" >0.13</td>
      <td id="T_4edac_row12_col30" class="data row12 col30" >0.60</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row13" class="row_heading level0 row13" >area_se</th>
      <td id="T_4edac_row13_col0" class="data row13 col0" >0.76</td>
      <td id="T_4edac_row13_col1" class="data row13 col1" >0.32</td>
      <td id="T_4edac_row13_col2" class="data row13 col2" >0.77</td>
      <td id="T_4edac_row13_col3" class="data row13 col3" >0.82</td>
      <td id="T_4edac_row13_col4" class="data row13 col4" >0.24</td>
      <td id="T_4edac_row13_col5" class="data row13 col5" >0.50</td>
      <td id="T_4edac_row13_col6" class="data row13 col6" >0.64</td>
      <td id="T_4edac_row13_col7" class="data row13 col7" >0.72</td>
      <td id="T_4edac_row13_col8" class="data row13 col8" >0.26</td>
      <td id="T_4edac_row13_col9" class="data row13 col9" >-0.09</td>
      <td id="T_4edac_row13_col10" class="data row13 col10" >0.95</td>
      <td id="T_4edac_row13_col11" class="data row13 col11" >0.12</td>
      <td id="T_4edac_row13_col12" class="data row13 col12" >0.93</td>
      <td id="T_4edac_row13_col13" class="data row13 col13" >1.00</td>
      <td id="T_4edac_row13_col14" class="data row13 col14" >0.07</td>
      <td id="T_4edac_row13_col15" class="data row13 col15" >0.32</td>
      <td id="T_4edac_row13_col16" class="data row13 col16" >0.27</td>
      <td id="T_4edac_row13_col17" class="data row13 col17" >0.44</td>
      <td id="T_4edac_row13_col18" class="data row13 col18" >0.09</td>
      <td id="T_4edac_row13_col19" class="data row13 col19" >0.12</td>
      <td id="T_4edac_row13_col20" class="data row13 col20" >0.80</td>
      <td id="T_4edac_row13_col21" class="data row13 col21" >0.27</td>
      <td id="T_4edac_row13_col22" class="data row13 col22" >0.80</td>
      <td id="T_4edac_row13_col23" class="data row13 col23" >0.85</td>
      <td id="T_4edac_row13_col24" class="data row13 col24" >0.16</td>
      <td id="T_4edac_row13_col25" class="data row13 col25" >0.35</td>
      <td id="T_4edac_row13_col26" class="data row13 col26" >0.44</td>
      <td id="T_4edac_row13_col27" class="data row13 col27" >0.59</td>
      <td id="T_4edac_row13_col28" class="data row13 col28" >0.13</td>
      <td id="T_4edac_row13_col29" class="data row13 col29" >0.06</td>
      <td id="T_4edac_row13_col30" class="data row13 col30" >0.59</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row14" class="row_heading level0 row14" >smoothness_se</th>
      <td id="T_4edac_row14_col0" class="data row14 col0" >-0.20</td>
      <td id="T_4edac_row14_col1" class="data row14 col1" >-0.03</td>
      <td id="T_4edac_row14_col2" class="data row14 col2" >-0.18</td>
      <td id="T_4edac_row14_col3" class="data row14 col3" >-0.15</td>
      <td id="T_4edac_row14_col4" class="data row14 col4" >0.32</td>
      <td id="T_4edac_row14_col5" class="data row14 col5" >0.14</td>
      <td id="T_4edac_row14_col6" class="data row14 col6" >0.14</td>
      <td id="T_4edac_row14_col7" class="data row14 col7" >0.06</td>
      <td id="T_4edac_row14_col8" class="data row14 col8" >0.17</td>
      <td id="T_4edac_row14_col9" class="data row14 col9" >0.37</td>
      <td id="T_4edac_row14_col10" class="data row14 col10" >0.17</td>
      <td id="T_4edac_row14_col11" class="data row14 col11" >0.38</td>
      <td id="T_4edac_row14_col12" class="data row14 col12" >0.16</td>
      <td id="T_4edac_row14_col13" class="data row14 col13" >0.07</td>
      <td id="T_4edac_row14_col14" class="data row14 col14" >1.00</td>
      <td id="T_4edac_row14_col15" class="data row14 col15" >0.39</td>
      <td id="T_4edac_row14_col16" class="data row14 col16" >0.32</td>
      <td id="T_4edac_row14_col17" class="data row14 col17" >0.42</td>
      <td id="T_4edac_row14_col18" class="data row14 col18" >0.41</td>
      <td id="T_4edac_row14_col19" class="data row14 col19" >0.45</td>
      <td id="T_4edac_row14_col20" class="data row14 col20" >-0.21</td>
      <td id="T_4edac_row14_col21" class="data row14 col21" >-0.11</td>
      <td id="T_4edac_row14_col22" class="data row14 col22" >-0.19</td>
      <td id="T_4edac_row14_col23" class="data row14 col23" >-0.16</td>
      <td id="T_4edac_row14_col24" class="data row14 col24" >0.29</td>
      <td id="T_4edac_row14_col25" class="data row14 col25" >-0.05</td>
      <td id="T_4edac_row14_col26" class="data row14 col26" >-0.02</td>
      <td id="T_4edac_row14_col27" class="data row14 col27" >-0.06</td>
      <td id="T_4edac_row14_col28" class="data row14 col28" >-0.12</td>
      <td id="T_4edac_row14_col29" class="data row14 col29" >0.06</td>
      <td id="T_4edac_row14_col30" class="data row14 col30" >-0.04</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row15" class="row_heading level0 row15" >compactness_se</th>
      <td id="T_4edac_row15_col0" class="data row15 col0" >0.21</td>
      <td id="T_4edac_row15_col1" class="data row15 col1" >0.19</td>
      <td id="T_4edac_row15_col2" class="data row15 col2" >0.25</td>
      <td id="T_4edac_row15_col3" class="data row15 col3" >0.22</td>
      <td id="T_4edac_row15_col4" class="data row15 col4" >0.31</td>
      <td id="T_4edac_row15_col5" class="data row15 col5" >0.73</td>
      <td id="T_4edac_row15_col6" class="data row15 col6" >0.68</td>
      <td id="T_4edac_row15_col7" class="data row15 col7" >0.49</td>
      <td id="T_4edac_row15_col8" class="data row15 col8" >0.40</td>
      <td id="T_4edac_row15_col9" class="data row15 col9" >0.55</td>
      <td id="T_4edac_row15_col10" class="data row15 col10" >0.39</td>
      <td id="T_4edac_row15_col11" class="data row15 col11" >0.26</td>
      <td id="T_4edac_row15_col12" class="data row15 col12" >0.46</td>
      <td id="T_4edac_row15_col13" class="data row15 col13" >0.32</td>
      <td id="T_4edac_row15_col14" class="data row15 col14" >0.39</td>
      <td id="T_4edac_row15_col15" class="data row15 col15" >1.00</td>
      <td id="T_4edac_row15_col16" class="data row15 col16" >0.79</td>
      <td id="T_4edac_row15_col17" class="data row15 col17" >0.76</td>
      <td id="T_4edac_row15_col18" class="data row15 col18" >0.41</td>
      <td id="T_4edac_row15_col19" class="data row15 col19" >0.79</td>
      <td id="T_4edac_row15_col20" class="data row15 col20" >0.20</td>
      <td id="T_4edac_row15_col21" class="data row15 col21" >0.14</td>
      <td id="T_4edac_row15_col22" class="data row15 col22" >0.26</td>
      <td id="T_4edac_row15_col23" class="data row15 col23" >0.20</td>
      <td id="T_4edac_row15_col24" class="data row15 col24" >0.19</td>
      <td id="T_4edac_row15_col25" class="data row15 col25" >0.66</td>
      <td id="T_4edac_row15_col26" class="data row15 col26" >0.63</td>
      <td id="T_4edac_row15_col27" class="data row15 col27" >0.46</td>
      <td id="T_4edac_row15_col28" class="data row15 col28" >0.24</td>
      <td id="T_4edac_row15_col29" class="data row15 col29" >0.58</td>
      <td id="T_4edac_row15_col30" class="data row15 col30" >0.28</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row16" class="row_heading level0 row16" >concavity_se</th>
      <td id="T_4edac_row16_col0" class="data row16 col0" >0.15</td>
      <td id="T_4edac_row16_col1" class="data row16 col1" >0.13</td>
      <td id="T_4edac_row16_col2" class="data row16 col2" >0.18</td>
      <td id="T_4edac_row16_col3" class="data row16 col3" >0.17</td>
      <td id="T_4edac_row16_col4" class="data row16 col4" >0.23</td>
      <td id="T_4edac_row16_col5" class="data row16 col5" >0.54</td>
      <td id="T_4edac_row16_col6" class="data row16 col6" >0.69</td>
      <td id="T_4edac_row16_col7" class="data row16 col7" >0.41</td>
      <td id="T_4edac_row16_col8" class="data row16 col8" >0.34</td>
      <td id="T_4edac_row16_col9" class="data row16 col9" >0.48</td>
      <td id="T_4edac_row16_col10" class="data row16 col10" >0.34</td>
      <td id="T_4edac_row16_col11" class="data row16 col11" >0.25</td>
      <td id="T_4edac_row16_col12" class="data row16 col12" >0.37</td>
      <td id="T_4edac_row16_col13" class="data row16 col13" >0.27</td>
      <td id="T_4edac_row16_col14" class="data row16 col14" >0.32</td>
      <td id="T_4edac_row16_col15" class="data row16 col15" >0.79</td>
      <td id="T_4edac_row16_col16" class="data row16 col16" >1.00</td>
      <td id="T_4edac_row16_col17" class="data row16 col17" >0.80</td>
      <td id="T_4edac_row16_col18" class="data row16 col18" >0.36</td>
      <td id="T_4edac_row16_col19" class="data row16 col19" >0.76</td>
      <td id="T_4edac_row16_col20" class="data row16 col20" >0.14</td>
      <td id="T_4edac_row16_col21" class="data row16 col21" >0.07</td>
      <td id="T_4edac_row16_col22" class="data row16 col22" >0.18</td>
      <td id="T_4edac_row16_col23" class="data row16 col23" >0.15</td>
      <td id="T_4edac_row16_col24" class="data row16 col24" >0.13</td>
      <td id="T_4edac_row16_col25" class="data row16 col25" >0.44</td>
      <td id="T_4edac_row16_col26" class="data row16 col26" >0.65</td>
      <td id="T_4edac_row16_col27" class="data row16 col27" >0.40</td>
      <td id="T_4edac_row16_col28" class="data row16 col28" >0.18</td>
      <td id="T_4edac_row16_col29" class="data row16 col29" >0.44</td>
      <td id="T_4edac_row16_col30" class="data row16 col30" >0.22</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row17" class="row_heading level0 row17" >concave points_se</th>
      <td id="T_4edac_row17_col0" class="data row17 col0" >0.32</td>
      <td id="T_4edac_row17_col1" class="data row17 col1" >0.18</td>
      <td id="T_4edac_row17_col2" class="data row17 col2" >0.35</td>
      <td id="T_4edac_row17_col3" class="data row17 col3" >0.33</td>
      <td id="T_4edac_row17_col4" class="data row17 col4" >0.36</td>
      <td id="T_4edac_row17_col5" class="data row17 col5" >0.62</td>
      <td id="T_4edac_row17_col6" class="data row17 col6" >0.70</td>
      <td id="T_4edac_row17_col7" class="data row17 col7" >0.59</td>
      <td id="T_4edac_row17_col8" class="data row17 col8" >0.36</td>
      <td id="T_4edac_row17_col9" class="data row17 col9" >0.40</td>
      <td id="T_4edac_row17_col10" class="data row17 col10" >0.55</td>
      <td id="T_4edac_row17_col11" class="data row17 col11" >0.30</td>
      <td id="T_4edac_row17_col12" class="data row17 col12" >0.59</td>
      <td id="T_4edac_row17_col13" class="data row17 col13" >0.44</td>
      <td id="T_4edac_row17_col14" class="data row17 col14" >0.42</td>
      <td id="T_4edac_row17_col15" class="data row17 col15" >0.76</td>
      <td id="T_4edac_row17_col16" class="data row17 col16" >0.80</td>
      <td id="T_4edac_row17_col17" class="data row17 col17" >1.00</td>
      <td id="T_4edac_row17_col18" class="data row17 col18" >0.36</td>
      <td id="T_4edac_row17_col19" class="data row17 col19" >0.67</td>
      <td id="T_4edac_row17_col20" class="data row17 col20" >0.30</td>
      <td id="T_4edac_row17_col21" class="data row17 col21" >0.08</td>
      <td id="T_4edac_row17_col22" class="data row17 col22" >0.34</td>
      <td id="T_4edac_row17_col23" class="data row17 col23" >0.31</td>
      <td id="T_4edac_row17_col24" class="data row17 col24" >0.19</td>
      <td id="T_4edac_row17_col25" class="data row17 col25" >0.44</td>
      <td id="T_4edac_row17_col26" class="data row17 col26" >0.57</td>
      <td id="T_4edac_row17_col27" class="data row17 col27" >0.56</td>
      <td id="T_4edac_row17_col28" class="data row17 col28" >0.09</td>
      <td id="T_4edac_row17_col29" class="data row17 col29" >0.34</td>
      <td id="T_4edac_row17_col30" class="data row17 col30" >0.38</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row18" class="row_heading level0 row18" >symmetry_se</th>
      <td id="T_4edac_row18_col0" class="data row18 col0" >-0.12</td>
      <td id="T_4edac_row18_col1" class="data row18 col1" >-0.00</td>
      <td id="T_4edac_row18_col2" class="data row18 col2" >-0.09</td>
      <td id="T_4edac_row18_col3" class="data row18 col3" >-0.09</td>
      <td id="T_4edac_row18_col4" class="data row18 col4" >0.22</td>
      <td id="T_4edac_row18_col5" class="data row18 col5" >0.24</td>
      <td id="T_4edac_row18_col6" class="data row18 col6" >0.22</td>
      <td id="T_4edac_row18_col7" class="data row18 col7" >0.11</td>
      <td id="T_4edac_row18_col8" class="data row18 col8" >0.45</td>
      <td id="T_4edac_row18_col9" class="data row18 col9" >0.35</td>
      <td id="T_4edac_row18_col10" class="data row18 col10" >0.20</td>
      <td id="T_4edac_row18_col11" class="data row18 col11" >0.40</td>
      <td id="T_4edac_row18_col12" class="data row18 col12" >0.22</td>
      <td id="T_4edac_row18_col13" class="data row18 col13" >0.09</td>
      <td id="T_4edac_row18_col14" class="data row18 col14" >0.41</td>
      <td id="T_4edac_row18_col15" class="data row18 col15" >0.41</td>
      <td id="T_4edac_row18_col16" class="data row18 col16" >0.36</td>
      <td id="T_4edac_row18_col17" class="data row18 col17" >0.36</td>
      <td id="T_4edac_row18_col18" class="data row18 col18" >1.00</td>
      <td id="T_4edac_row18_col19" class="data row18 col19" >0.36</td>
      <td id="T_4edac_row18_col20" class="data row18 col20" >-0.13</td>
      <td id="T_4edac_row18_col21" class="data row18 col21" >-0.08</td>
      <td id="T_4edac_row18_col22" class="data row18 col22" >-0.11</td>
      <td id="T_4edac_row18_col23" class="data row18 col23" >-0.11</td>
      <td id="T_4edac_row18_col24" class="data row18 col24" >-0.00</td>
      <td id="T_4edac_row18_col25" class="data row18 col25" >0.08</td>
      <td id="T_4edac_row18_col26" class="data row18 col26" >0.11</td>
      <td id="T_4edac_row18_col27" class="data row18 col27" >0.01</td>
      <td id="T_4edac_row18_col28" class="data row18 col28" >0.40</td>
      <td id="T_4edac_row18_col29" class="data row18 col29" >0.09</td>
      <td id="T_4edac_row18_col30" class="data row18 col30" >-0.00</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row19" class="row_heading level0 row19" >fractal_dimension_se</th>
      <td id="T_4edac_row19_col0" class="data row19 col0" >-0.06</td>
      <td id="T_4edac_row19_col1" class="data row19 col1" >0.03</td>
      <td id="T_4edac_row19_col2" class="data row19 col2" >-0.02</td>
      <td id="T_4edac_row19_col3" class="data row19 col3" >-0.03</td>
      <td id="T_4edac_row19_col4" class="data row19 col4" >0.25</td>
      <td id="T_4edac_row19_col5" class="data row19 col5" >0.47</td>
      <td id="T_4edac_row19_col6" class="data row19 col6" >0.47</td>
      <td id="T_4edac_row19_col7" class="data row19 col7" >0.25</td>
      <td id="T_4edac_row19_col8" class="data row19 col8" >0.31</td>
      <td id="T_4edac_row19_col9" class="data row19 col9" >0.68</td>
      <td id="T_4edac_row19_col10" class="data row19 col10" >0.22</td>
      <td id="T_4edac_row19_col11" class="data row19 col11" >0.30</td>
      <td id="T_4edac_row19_col12" class="data row19 col12" >0.25</td>
      <td id="T_4edac_row19_col13" class="data row19 col13" >0.12</td>
      <td id="T_4edac_row19_col14" class="data row19 col14" >0.45</td>
      <td id="T_4edac_row19_col15" class="data row19 col15" >0.79</td>
      <td id="T_4edac_row19_col16" class="data row19 col16" >0.76</td>
      <td id="T_4edac_row19_col17" class="data row19 col17" >0.67</td>
      <td id="T_4edac_row19_col18" class="data row19 col18" >0.36</td>
      <td id="T_4edac_row19_col19" class="data row19 col19" >1.00</td>
      <td id="T_4edac_row19_col20" class="data row19 col20" >-0.05</td>
      <td id="T_4edac_row19_col21" class="data row19 col21" >-0.03</td>
      <td id="T_4edac_row19_col22" class="data row19 col22" >-0.02</td>
      <td id="T_4edac_row19_col23" class="data row19 col23" >-0.03</td>
      <td id="T_4edac_row19_col24" class="data row19 col24" >0.10</td>
      <td id="T_4edac_row19_col25" class="data row19 col25" >0.36</td>
      <td id="T_4edac_row19_col26" class="data row19 col26" >0.39</td>
      <td id="T_4edac_row19_col27" class="data row19 col27" >0.21</td>
      <td id="T_4edac_row19_col28" class="data row19 col28" >0.07</td>
      <td id="T_4edac_row19_col29" class="data row19 col29" >0.57</td>
      <td id="T_4edac_row19_col30" class="data row19 col30" >0.06</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row20" class="row_heading level0 row20" >radius_worst</th>
      <td id="T_4edac_row20_col0" class="data row20 col0" >0.97</td>
      <td id="T_4edac_row20_col1" class="data row20 col1" >0.41</td>
      <td id="T_4edac_row20_col2" class="data row20 col2" >0.97</td>
      <td id="T_4edac_row20_col3" class="data row20 col3" >0.97</td>
      <td id="T_4edac_row20_col4" class="data row20 col4" >0.20</td>
      <td id="T_4edac_row20_col5" class="data row20 col5" >0.55</td>
      <td id="T_4edac_row20_col6" class="data row20 col6" >0.67</td>
      <td id="T_4edac_row20_col7" class="data row20 col7" >0.82</td>
      <td id="T_4edac_row20_col8" class="data row20 col8" >0.20</td>
      <td id="T_4edac_row20_col9" class="data row20 col9" >-0.25</td>
      <td id="T_4edac_row20_col10" class="data row20 col10" >0.75</td>
      <td id="T_4edac_row20_col11" class="data row20 col11" >-0.10</td>
      <td id="T_4edac_row20_col12" class="data row20 col12" >0.73</td>
      <td id="T_4edac_row20_col13" class="data row20 col13" >0.80</td>
      <td id="T_4edac_row20_col14" class="data row20 col14" >-0.21</td>
      <td id="T_4edac_row20_col15" class="data row20 col15" >0.20</td>
      <td id="T_4edac_row20_col16" class="data row20 col16" >0.14</td>
      <td id="T_4edac_row20_col17" class="data row20 col17" >0.30</td>
      <td id="T_4edac_row20_col18" class="data row20 col18" >-0.13</td>
      <td id="T_4edac_row20_col19" class="data row20 col19" >-0.05</td>
      <td id="T_4edac_row20_col20" class="data row20 col20" >1.00</td>
      <td id="T_4edac_row20_col21" class="data row20 col21" >0.42</td>
      <td id="T_4edac_row20_col22" class="data row20 col22" >0.99</td>
      <td id="T_4edac_row20_col23" class="data row20 col23" >0.98</td>
      <td id="T_4edac_row20_col24" class="data row20 col24" >0.24</td>
      <td id="T_4edac_row20_col25" class="data row20 col25" >0.50</td>
      <td id="T_4edac_row20_col26" class="data row20 col26" >0.57</td>
      <td id="T_4edac_row20_col27" class="data row20 col27" >0.78</td>
      <td id="T_4edac_row20_col28" class="data row20 col28" >0.27</td>
      <td id="T_4edac_row20_col29" class="data row20 col29" >0.12</td>
      <td id="T_4edac_row20_col30" class="data row20 col30" >0.79</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row21" class="row_heading level0 row21" >texture_worst</th>
      <td id="T_4edac_row21_col0" class="data row21 col0" >0.36</td>
      <td id="T_4edac_row21_col1" class="data row21 col1" >0.92</td>
      <td id="T_4edac_row21_col2" class="data row21 col2" >0.37</td>
      <td id="T_4edac_row21_col3" class="data row21 col3" >0.35</td>
      <td id="T_4edac_row21_col4" class="data row21 col4" >0.02</td>
      <td id="T_4edac_row21_col5" class="data row21 col5" >0.28</td>
      <td id="T_4edac_row21_col6" class="data row21 col6" >0.32</td>
      <td id="T_4edac_row21_col7" class="data row21 col7" >0.34</td>
      <td id="T_4edac_row21_col8" class="data row21 col8" >0.10</td>
      <td id="T_4edac_row21_col9" class="data row21 col9" >-0.08</td>
      <td id="T_4edac_row21_col10" class="data row21 col10" >0.25</td>
      <td id="T_4edac_row21_col11" class="data row21 col11" >0.38</td>
      <td id="T_4edac_row21_col12" class="data row21 col12" >0.26</td>
      <td id="T_4edac_row21_col13" class="data row21 col13" >0.27</td>
      <td id="T_4edac_row21_col14" class="data row21 col14" >-0.11</td>
      <td id="T_4edac_row21_col15" class="data row21 col15" >0.14</td>
      <td id="T_4edac_row21_col16" class="data row21 col16" >0.07</td>
      <td id="T_4edac_row21_col17" class="data row21 col17" >0.08</td>
      <td id="T_4edac_row21_col18" class="data row21 col18" >-0.08</td>
      <td id="T_4edac_row21_col19" class="data row21 col19" >-0.03</td>
      <td id="T_4edac_row21_col20" class="data row21 col20" >0.42</td>
      <td id="T_4edac_row21_col21" class="data row21 col21" >1.00</td>
      <td id="T_4edac_row21_col22" class="data row21 col22" >0.42</td>
      <td id="T_4edac_row21_col23" class="data row21 col23" >0.40</td>
      <td id="T_4edac_row21_col24" class="data row21 col24" >0.21</td>
      <td id="T_4edac_row21_col25" class="data row21 col25" >0.39</td>
      <td id="T_4edac_row21_col26" class="data row21 col26" >0.38</td>
      <td id="T_4edac_row21_col27" class="data row21 col27" >0.40</td>
      <td id="T_4edac_row21_col28" class="data row21 col28" >0.24</td>
      <td id="T_4edac_row21_col29" class="data row21 col29" >0.23</td>
      <td id="T_4edac_row21_col30" class="data row21 col30" >0.49</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row22" class="row_heading level0 row22" >perimeter_worst</th>
      <td id="T_4edac_row22_col0" class="data row22 col0" >0.97</td>
      <td id="T_4edac_row22_col1" class="data row22 col1" >0.42</td>
      <td id="T_4edac_row22_col2" class="data row22 col2" >0.97</td>
      <td id="T_4edac_row22_col3" class="data row22 col3" >0.97</td>
      <td id="T_4edac_row22_col4" class="data row22 col4" >0.23</td>
      <td id="T_4edac_row22_col5" class="data row22 col5" >0.61</td>
      <td id="T_4edac_row22_col6" class="data row22 col6" >0.71</td>
      <td id="T_4edac_row22_col7" class="data row22 col7" >0.85</td>
      <td id="T_4edac_row22_col8" class="data row22 col8" >0.23</td>
      <td id="T_4edac_row22_col9" class="data row22 col9" >-0.20</td>
      <td id="T_4edac_row22_col10" class="data row22 col10" >0.75</td>
      <td id="T_4edac_row22_col11" class="data row22 col11" >-0.09</td>
      <td id="T_4edac_row22_col12" class="data row22 col12" >0.76</td>
      <td id="T_4edac_row22_col13" class="data row22 col13" >0.80</td>
      <td id="T_4edac_row22_col14" class="data row22 col14" >-0.19</td>
      <td id="T_4edac_row22_col15" class="data row22 col15" >0.26</td>
      <td id="T_4edac_row22_col16" class="data row22 col16" >0.18</td>
      <td id="T_4edac_row22_col17" class="data row22 col17" >0.34</td>
      <td id="T_4edac_row22_col18" class="data row22 col18" >-0.11</td>
      <td id="T_4edac_row22_col19" class="data row22 col19" >-0.02</td>
      <td id="T_4edac_row22_col20" class="data row22 col20" >0.99</td>
      <td id="T_4edac_row22_col21" class="data row22 col21" >0.42</td>
      <td id="T_4edac_row22_col22" class="data row22 col22" >1.00</td>
      <td id="T_4edac_row22_col23" class="data row22 col23" >0.98</td>
      <td id="T_4edac_row22_col24" class="data row22 col24" >0.26</td>
      <td id="T_4edac_row22_col25" class="data row22 col25" >0.56</td>
      <td id="T_4edac_row22_col26" class="data row22 col26" >0.62</td>
      <td id="T_4edac_row22_col27" class="data row22 col27" >0.81</td>
      <td id="T_4edac_row22_col28" class="data row22 col28" >0.29</td>
      <td id="T_4edac_row22_col29" class="data row22 col29" >0.17</td>
      <td id="T_4edac_row22_col30" class="data row22 col30" >0.79</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row23" class="row_heading level0 row23" >area_worst</th>
      <td id="T_4edac_row23_col0" class="data row23 col0" >0.94</td>
      <td id="T_4edac_row23_col1" class="data row23 col1" >0.40</td>
      <td id="T_4edac_row23_col2" class="data row23 col2" >0.94</td>
      <td id="T_4edac_row23_col3" class="data row23 col3" >0.97</td>
      <td id="T_4edac_row23_col4" class="data row23 col4" >0.20</td>
      <td id="T_4edac_row23_col5" class="data row23 col5" >0.53</td>
      <td id="T_4edac_row23_col6" class="data row23 col6" >0.66</td>
      <td id="T_4edac_row23_col7" class="data row23 col7" >0.81</td>
      <td id="T_4edac_row23_col8" class="data row23 col8" >0.20</td>
      <td id="T_4edac_row23_col9" class="data row23 col9" >-0.22</td>
      <td id="T_4edac_row23_col10" class="data row23 col10" >0.78</td>
      <td id="T_4edac_row23_col11" class="data row23 col11" >-0.07</td>
      <td id="T_4edac_row23_col12" class="data row23 col12" >0.77</td>
      <td id="T_4edac_row23_col13" class="data row23 col13" >0.85</td>
      <td id="T_4edac_row23_col14" class="data row23 col14" >-0.16</td>
      <td id="T_4edac_row23_col15" class="data row23 col15" >0.20</td>
      <td id="T_4edac_row23_col16" class="data row23 col16" >0.15</td>
      <td id="T_4edac_row23_col17" class="data row23 col17" >0.31</td>
      <td id="T_4edac_row23_col18" class="data row23 col18" >-0.11</td>
      <td id="T_4edac_row23_col19" class="data row23 col19" >-0.03</td>
      <td id="T_4edac_row23_col20" class="data row23 col20" >0.98</td>
      <td id="T_4edac_row23_col21" class="data row23 col21" >0.40</td>
      <td id="T_4edac_row23_col22" class="data row23 col22" >0.98</td>
      <td id="T_4edac_row23_col23" class="data row23 col23" >1.00</td>
      <td id="T_4edac_row23_col24" class="data row23 col24" >0.23</td>
      <td id="T_4edac_row23_col25" class="data row23 col25" >0.47</td>
      <td id="T_4edac_row23_col26" class="data row23 col26" >0.55</td>
      <td id="T_4edac_row23_col27" class="data row23 col27" >0.75</td>
      <td id="T_4edac_row23_col28" class="data row23 col28" >0.24</td>
      <td id="T_4edac_row23_col29" class="data row23 col29" >0.11</td>
      <td id="T_4edac_row23_col30" class="data row23 col30" >0.75</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row24" class="row_heading level0 row24" >smoothness_worst</th>
      <td id="T_4edac_row24_col0" class="data row24 col0" >0.15</td>
      <td id="T_4edac_row24_col1" class="data row24 col1" >0.05</td>
      <td id="T_4edac_row24_col2" class="data row24 col2" >0.18</td>
      <td id="T_4edac_row24_col3" class="data row24 col3" >0.15</td>
      <td id="T_4edac_row24_col4" class="data row24 col4" >0.79</td>
      <td id="T_4edac_row24_col5" class="data row24 col5" >0.54</td>
      <td id="T_4edac_row24_col6" class="data row24 col6" >0.42</td>
      <td id="T_4edac_row24_col7" class="data row24 col7" >0.45</td>
      <td id="T_4edac_row24_col8" class="data row24 col8" >0.42</td>
      <td id="T_4edac_row24_col9" class="data row24 col9" >0.45</td>
      <td id="T_4edac_row24_col10" class="data row24 col10" >0.17</td>
      <td id="T_4edac_row24_col11" class="data row24 col11" >-0.07</td>
      <td id="T_4edac_row24_col12" class="data row24 col12" >0.18</td>
      <td id="T_4edac_row24_col13" class="data row24 col13" >0.16</td>
      <td id="T_4edac_row24_col14" class="data row24 col14" >0.29</td>
      <td id="T_4edac_row24_col15" class="data row24 col15" >0.19</td>
      <td id="T_4edac_row24_col16" class="data row24 col16" >0.13</td>
      <td id="T_4edac_row24_col17" class="data row24 col17" >0.19</td>
      <td id="T_4edac_row24_col18" class="data row24 col18" >-0.00</td>
      <td id="T_4edac_row24_col19" class="data row24 col19" >0.10</td>
      <td id="T_4edac_row24_col20" class="data row24 col20" >0.24</td>
      <td id="T_4edac_row24_col21" class="data row24 col21" >0.21</td>
      <td id="T_4edac_row24_col22" class="data row24 col22" >0.26</td>
      <td id="T_4edac_row24_col23" class="data row24 col23" >0.23</td>
      <td id="T_4edac_row24_col24" class="data row24 col24" >1.00</td>
      <td id="T_4edac_row24_col25" class="data row24 col25" >0.54</td>
      <td id="T_4edac_row24_col26" class="data row24 col26" >0.48</td>
      <td id="T_4edac_row24_col27" class="data row24 col27" >0.54</td>
      <td id="T_4edac_row24_col28" class="data row24 col28" >0.48</td>
      <td id="T_4edac_row24_col29" class="data row24 col29" >0.56</td>
      <td id="T_4edac_row24_col30" class="data row24 col30" >0.41</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row25" class="row_heading level0 row25" >compactness_worst</th>
      <td id="T_4edac_row25_col0" class="data row25 col0" >0.45</td>
      <td id="T_4edac_row25_col1" class="data row25 col1" >0.32</td>
      <td id="T_4edac_row25_col2" class="data row25 col2" >0.49</td>
      <td id="T_4edac_row25_col3" class="data row25 col3" >0.43</td>
      <td id="T_4edac_row25_col4" class="data row25 col4" >0.45</td>
      <td id="T_4edac_row25_col5" class="data row25 col5" >0.88</td>
      <td id="T_4edac_row25_col6" class="data row25 col6" >0.75</td>
      <td id="T_4edac_row25_col7" class="data row25 col7" >0.68</td>
      <td id="T_4edac_row25_col8" class="data row25 col8" >0.48</td>
      <td id="T_4edac_row25_col9" class="data row25 col9" >0.43</td>
      <td id="T_4edac_row25_col10" class="data row25 col10" >0.34</td>
      <td id="T_4edac_row25_col11" class="data row25 col11" >-0.07</td>
      <td id="T_4edac_row25_col12" class="data row25 col12" >0.42</td>
      <td id="T_4edac_row25_col13" class="data row25 col13" >0.35</td>
      <td id="T_4edac_row25_col14" class="data row25 col14" >-0.05</td>
      <td id="T_4edac_row25_col15" class="data row25 col15" >0.66</td>
      <td id="T_4edac_row25_col16" class="data row25 col16" >0.44</td>
      <td id="T_4edac_row25_col17" class="data row25 col17" >0.44</td>
      <td id="T_4edac_row25_col18" class="data row25 col18" >0.08</td>
      <td id="T_4edac_row25_col19" class="data row25 col19" >0.36</td>
      <td id="T_4edac_row25_col20" class="data row25 col20" >0.50</td>
      <td id="T_4edac_row25_col21" class="data row25 col21" >0.39</td>
      <td id="T_4edac_row25_col22" class="data row25 col22" >0.56</td>
      <td id="T_4edac_row25_col23" class="data row25 col23" >0.47</td>
      <td id="T_4edac_row25_col24" class="data row25 col24" >0.54</td>
      <td id="T_4edac_row25_col25" class="data row25 col25" >1.00</td>
      <td id="T_4edac_row25_col26" class="data row25 col26" >0.88</td>
      <td id="T_4edac_row25_col27" class="data row25 col27" >0.80</td>
      <td id="T_4edac_row25_col28" class="data row25 col28" >0.60</td>
      <td id="T_4edac_row25_col29" class="data row25 col29" >0.81</td>
      <td id="T_4edac_row25_col30" class="data row25 col30" >0.59</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row26" class="row_heading level0 row26" >concavity_worst</th>
      <td id="T_4edac_row26_col0" class="data row26 col0" >0.53</td>
      <td id="T_4edac_row26_col1" class="data row26 col1" >0.33</td>
      <td id="T_4edac_row26_col2" class="data row26 col2" >0.56</td>
      <td id="T_4edac_row26_col3" class="data row26 col3" >0.52</td>
      <td id="T_4edac_row26_col4" class="data row26 col4" >0.41</td>
      <td id="T_4edac_row26_col5" class="data row26 col5" >0.81</td>
      <td id="T_4edac_row26_col6" class="data row26 col6" >0.89</td>
      <td id="T_4edac_row26_col7" class="data row26 col7" >0.76</td>
      <td id="T_4edac_row26_col8" class="data row26 col8" >0.46</td>
      <td id="T_4edac_row26_col9" class="data row26 col9" >0.35</td>
      <td id="T_4edac_row26_col10" class="data row26 col10" >0.44</td>
      <td id="T_4edac_row26_col11" class="data row26 col11" >-0.01</td>
      <td id="T_4edac_row26_col12" class="data row26 col12" >0.49</td>
      <td id="T_4edac_row26_col13" class="data row26 col13" >0.44</td>
      <td id="T_4edac_row26_col14" class="data row26 col14" >-0.02</td>
      <td id="T_4edac_row26_col15" class="data row26 col15" >0.63</td>
      <td id="T_4edac_row26_col16" class="data row26 col16" >0.65</td>
      <td id="T_4edac_row26_col17" class="data row26 col17" >0.57</td>
      <td id="T_4edac_row26_col18" class="data row26 col18" >0.11</td>
      <td id="T_4edac_row26_col19" class="data row26 col19" >0.39</td>
      <td id="T_4edac_row26_col20" class="data row26 col20" >0.57</td>
      <td id="T_4edac_row26_col21" class="data row26 col21" >0.38</td>
      <td id="T_4edac_row26_col22" class="data row26 col22" >0.62</td>
      <td id="T_4edac_row26_col23" class="data row26 col23" >0.55</td>
      <td id="T_4edac_row26_col24" class="data row26 col24" >0.48</td>
      <td id="T_4edac_row26_col25" class="data row26 col25" >0.88</td>
      <td id="T_4edac_row26_col26" class="data row26 col26" >1.00</td>
      <td id="T_4edac_row26_col27" class="data row26 col27" >0.86</td>
      <td id="T_4edac_row26_col28" class="data row26 col28" >0.53</td>
      <td id="T_4edac_row26_col29" class="data row26 col29" >0.68</td>
      <td id="T_4edac_row26_col30" class="data row26 col30" >0.64</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row27" class="row_heading level0 row27" >concave points_worst</th>
      <td id="T_4edac_row27_col0" class="data row27 col0" >0.74</td>
      <td id="T_4edac_row27_col1" class="data row27 col1" >0.35</td>
      <td id="T_4edac_row27_col2" class="data row27 col2" >0.76</td>
      <td id="T_4edac_row27_col3" class="data row27 col3" >0.73</td>
      <td id="T_4edac_row27_col4" class="data row27 col4" >0.47</td>
      <td id="T_4edac_row27_col5" class="data row27 col5" >0.81</td>
      <td id="T_4edac_row27_col6" class="data row27 col6" >0.86</td>
      <td id="T_4edac_row27_col7" class="data row27 col7" >0.91</td>
      <td id="T_4edac_row27_col8" class="data row27 col8" >0.43</td>
      <td id="T_4edac_row27_col9" class="data row27 col9" >0.19</td>
      <td id="T_4edac_row27_col10" class="data row27 col10" >0.58</td>
      <td id="T_4edac_row27_col11" class="data row27 col11" >-0.05</td>
      <td id="T_4edac_row27_col12" class="data row27 col12" >0.62</td>
      <td id="T_4edac_row27_col13" class="data row27 col13" >0.59</td>
      <td id="T_4edac_row27_col14" class="data row27 col14" >-0.06</td>
      <td id="T_4edac_row27_col15" class="data row27 col15" >0.46</td>
      <td id="T_4edac_row27_col16" class="data row27 col16" >0.40</td>
      <td id="T_4edac_row27_col17" class="data row27 col17" >0.56</td>
      <td id="T_4edac_row27_col18" class="data row27 col18" >0.01</td>
      <td id="T_4edac_row27_col19" class="data row27 col19" >0.21</td>
      <td id="T_4edac_row27_col20" class="data row27 col20" >0.78</td>
      <td id="T_4edac_row27_col21" class="data row27 col21" >0.40</td>
      <td id="T_4edac_row27_col22" class="data row27 col22" >0.81</td>
      <td id="T_4edac_row27_col23" class="data row27 col23" >0.75</td>
      <td id="T_4edac_row27_col24" class="data row27 col24" >0.54</td>
      <td id="T_4edac_row27_col25" class="data row27 col25" >0.80</td>
      <td id="T_4edac_row27_col26" class="data row27 col26" >0.86</td>
      <td id="T_4edac_row27_col27" class="data row27 col27" >1.00</td>
      <td id="T_4edac_row27_col28" class="data row27 col28" >0.49</td>
      <td id="T_4edac_row27_col29" class="data row27 col29" >0.54</td>
      <td id="T_4edac_row27_col30" class="data row27 col30" >0.78</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row28" class="row_heading level0 row28" >symmetry_worst</th>
      <td id="T_4edac_row28_col0" class="data row28 col0" >0.20</td>
      <td id="T_4edac_row28_col1" class="data row28 col1" >0.12</td>
      <td id="T_4edac_row28_col2" class="data row28 col2" >0.22</td>
      <td id="T_4edac_row28_col3" class="data row28 col3" >0.18</td>
      <td id="T_4edac_row28_col4" class="data row28 col4" >0.39</td>
      <td id="T_4edac_row28_col5" class="data row28 col5" >0.50</td>
      <td id="T_4edac_row28_col6" class="data row28 col6" >0.40</td>
      <td id="T_4edac_row28_col7" class="data row28 col7" >0.37</td>
      <td id="T_4edac_row28_col8" class="data row28 col8" >0.69</td>
      <td id="T_4edac_row28_col9" class="data row28 col9" >0.28</td>
      <td id="T_4edac_row28_col10" class="data row28 col10" >0.13</td>
      <td id="T_4edac_row28_col11" class="data row28 col11" >-0.14</td>
      <td id="T_4edac_row28_col12" class="data row28 col12" >0.14</td>
      <td id="T_4edac_row28_col13" class="data row28 col13" >0.13</td>
      <td id="T_4edac_row28_col14" class="data row28 col14" >-0.12</td>
      <td id="T_4edac_row28_col15" class="data row28 col15" >0.24</td>
      <td id="T_4edac_row28_col16" class="data row28 col16" >0.18</td>
      <td id="T_4edac_row28_col17" class="data row28 col17" >0.09</td>
      <td id="T_4edac_row28_col18" class="data row28 col18" >0.40</td>
      <td id="T_4edac_row28_col19" class="data row28 col19" >0.07</td>
      <td id="T_4edac_row28_col20" class="data row28 col20" >0.27</td>
      <td id="T_4edac_row28_col21" class="data row28 col21" >0.24</td>
      <td id="T_4edac_row28_col22" class="data row28 col22" >0.29</td>
      <td id="T_4edac_row28_col23" class="data row28 col23" >0.24</td>
      <td id="T_4edac_row28_col24" class="data row28 col24" >0.48</td>
      <td id="T_4edac_row28_col25" class="data row28 col25" >0.60</td>
      <td id="T_4edac_row28_col26" class="data row28 col26" >0.53</td>
      <td id="T_4edac_row28_col27" class="data row28 col27" >0.49</td>
      <td id="T_4edac_row28_col28" class="data row28 col28" >1.00</td>
      <td id="T_4edac_row28_col29" class="data row28 col29" >0.52</td>
      <td id="T_4edac_row28_col30" class="data row28 col30" >0.41</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row29" class="row_heading level0 row29" >fractal_dimension_worst</th>
      <td id="T_4edac_row29_col0" class="data row29 col0" >0.04</td>
      <td id="T_4edac_row29_col1" class="data row29 col1" >0.12</td>
      <td id="T_4edac_row29_col2" class="data row29 col2" >0.09</td>
      <td id="T_4edac_row29_col3" class="data row29 col3" >0.04</td>
      <td id="T_4edac_row29_col4" class="data row29 col4" >0.47</td>
      <td id="T_4edac_row29_col5" class="data row29 col5" >0.69</td>
      <td id="T_4edac_row29_col6" class="data row29 col6" >0.53</td>
      <td id="T_4edac_row29_col7" class="data row29 col7" >0.40</td>
      <td id="T_4edac_row29_col8" class="data row29 col8" >0.44</td>
      <td id="T_4edac_row29_col9" class="data row29 col9" >0.75</td>
      <td id="T_4edac_row29_col10" class="data row29 col10" >0.08</td>
      <td id="T_4edac_row29_col11" class="data row29 col11" >-0.04</td>
      <td id="T_4edac_row29_col12" class="data row29 col12" >0.13</td>
      <td id="T_4edac_row29_col13" class="data row29 col13" >0.06</td>
      <td id="T_4edac_row29_col14" class="data row29 col14" >0.06</td>
      <td id="T_4edac_row29_col15" class="data row29 col15" >0.58</td>
      <td id="T_4edac_row29_col16" class="data row29 col16" >0.44</td>
      <td id="T_4edac_row29_col17" class="data row29 col17" >0.34</td>
      <td id="T_4edac_row29_col18" class="data row29 col18" >0.09</td>
      <td id="T_4edac_row29_col19" class="data row29 col19" >0.57</td>
      <td id="T_4edac_row29_col20" class="data row29 col20" >0.12</td>
      <td id="T_4edac_row29_col21" class="data row29 col21" >0.23</td>
      <td id="T_4edac_row29_col22" class="data row29 col22" >0.17</td>
      <td id="T_4edac_row29_col23" class="data row29 col23" >0.11</td>
      <td id="T_4edac_row29_col24" class="data row29 col24" >0.56</td>
      <td id="T_4edac_row29_col25" class="data row29 col25" >0.81</td>
      <td id="T_4edac_row29_col26" class="data row29 col26" >0.68</td>
      <td id="T_4edac_row29_col27" class="data row29 col27" >0.54</td>
      <td id="T_4edac_row29_col28" class="data row29 col28" >0.52</td>
      <td id="T_4edac_row29_col29" class="data row29 col29" >1.00</td>
      <td id="T_4edac_row29_col30" class="data row29 col30" >0.31</td>
    </tr>
    <tr>
      <th id="T_4edac_level0_row30" class="row_heading level0 row30" >diagnosis</th>
      <td id="T_4edac_row30_col0" class="data row30 col0" >0.74</td>
      <td id="T_4edac_row30_col1" class="data row30 col1" >0.46</td>
      <td id="T_4edac_row30_col2" class="data row30 col2" >0.75</td>
      <td id="T_4edac_row30_col3" class="data row30 col3" >0.73</td>
      <td id="T_4edac_row30_col4" class="data row30 col4" >0.33</td>
      <td id="T_4edac_row30_col5" class="data row30 col5" >0.59</td>
      <td id="T_4edac_row30_col6" class="data row30 col6" >0.67</td>
      <td id="T_4edac_row30_col7" class="data row30 col7" >0.77</td>
      <td id="T_4edac_row30_col8" class="data row30 col8" >0.33</td>
      <td id="T_4edac_row30_col9" class="data row30 col9" >-0.05</td>
      <td id="T_4edac_row30_col10" class="data row30 col10" >0.61</td>
      <td id="T_4edac_row30_col11" class="data row30 col11" >0.01</td>
      <td id="T_4edac_row30_col12" class="data row30 col12" >0.60</td>
      <td id="T_4edac_row30_col13" class="data row30 col13" >0.59</td>
      <td id="T_4edac_row30_col14" class="data row30 col14" >-0.04</td>
      <td id="T_4edac_row30_col15" class="data row30 col15" >0.28</td>
      <td id="T_4edac_row30_col16" class="data row30 col16" >0.22</td>
      <td id="T_4edac_row30_col17" class="data row30 col17" >0.38</td>
      <td id="T_4edac_row30_col18" class="data row30 col18" >-0.00</td>
      <td id="T_4edac_row30_col19" class="data row30 col19" >0.06</td>
      <td id="T_4edac_row30_col20" class="data row30 col20" >0.79</td>
      <td id="T_4edac_row30_col21" class="data row30 col21" >0.49</td>
      <td id="T_4edac_row30_col22" class="data row30 col22" >0.79</td>
      <td id="T_4edac_row30_col23" class="data row30 col23" >0.75</td>
      <td id="T_4edac_row30_col24" class="data row30 col24" >0.41</td>
      <td id="T_4edac_row30_col25" class="data row30 col25" >0.59</td>
      <td id="T_4edac_row30_col26" class="data row30 col26" >0.64</td>
      <td id="T_4edac_row30_col27" class="data row30 col27" >0.78</td>
      <td id="T_4edac_row30_col28" class="data row30 col28" >0.41</td>
      <td id="T_4edac_row30_col29" class="data row30 col29" >0.31</td>
      <td id="T_4edac_row30_col30" class="data row30 col30" >1.00</td>
    </tr>
  </tbody>
</table>


**Number of original features:** 30

**Number retained features:** 12

In [17]:
# Train a decision tree using X_train_filtered and X_test_filtered
table = Table()

parameters = {
  "table":table, "algorithm": "Decision Tree", "hyperparameter": "feature selection",
  "X":X_train, "Y":Y_train, "testX" : X_test, "testY": Y_test
}
trainAndPredictDecisionTree(**parameters, value="baseline", max_depth=None)

parameters = {
  "table":table, "algorithm": "Decision Tree", "hyperparameter": "feature selection",
  "X":X_train_filtered, "Y":Y_train, "testX" : X_test_filtered, "testY": Y_test
}
trainAndPredictDecisionTree(**parameters, value="pearson", max_depth=None)

table.tableFooter()

|Algorithm| Hyperparameter | Value | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- | --- |
| Decision Tree | feature selection | baseline | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
|  |  | pearson | 0.9415 | 0.9419 | 0.9415 | 0.9412 |


#### Pearson Correlation Performance Evaluation

Using Pearson Correlation for feature selection reduced the number of features from 30 to 12. It had minimal impact on the resulting performance. The drop in precision was a single mis-predicted positive. Looking at the depth of the resulting trees, the baseline tree had a max depth of 5, while the filtered tree had a depth of 8. A shallower tree will perform better - fewer operations to classify an instance.

### 3.b Feature Extraction using PCA

In [18]:
table = Table()

max_depth = [2, 8, None]

parameters = {
  "table":table, "algorithm": "Baseline Decision Tree", "hyperparameter": "max_depth",
  "X":X_train_standardized, "Y":Y_train, "testX" : X_test_standardized, "testY": Y_test
}
for n in max_depth:
    trainAndPredictDecisionTree(**parameters, value=n, max_depth=n)

pca = PCA()
pca.fit(X_train_standardized)
X_train_pca = pca.transform(X_train_standardized)
X_test_pca = pca.transform(X_test_standardized)

parameters |= {
   "algorithm":"PCA Decision Tree",
   "X":X_train_pca, "Y":Y_train, "testX" : X_test_pca, "testY": Y_test
}
for n in max_depth:
    trainAndPredictDecisionTree(**parameters, value=n, max_depth=n)

table.tableFooter()



|Algorithm| Hyperparameter | Value | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- | --- |
| Baseline Decision Tree | max_depth | 2 | 0.883 | 0.8841 | 0.883 | 0.8815 |
|  |  | 8 | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
|  |  | None | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
| PCA Decision Tree | max_depth | 2 | 0.9123 | 0.9136 | 0.9123 | 0.9113 |
|  |  | 8 | 0.9415 | 0.9415 | 0.9415 | 0.9415 |
|  |  | None | 0.9415 | 0.9415 | 0.9415 | 0.9415 |


#### PCA Performance Evaluation

PCA performs similarly to the baseline tree. Both peak at the same values. However, PCA performs better at the minimum tree size. It might be interesting to see how PCA + AdaBoost perform. The baseline tree has a higher precision, indicating fewer false positives. The initial peak hints that since the features are orthogonal, more information is available earlier in the tree.

## 4. Comparison: PCA vs Feature Selection

In [19]:
table = Table()

parameters = {
  "table": table, "algorithm": "Decision Tree", "hyperparameter": "feature selection",
  "X":X_train_standardized, "Y":Y_train, "testX" : X_test_standardized, "testY": Y_test
}
trainAndPredictDecisionTree(**parameters, value="baseline", max_depth=None)


parameters |= {
    "algorithm": "Decision Tree",
    "X": X_train_filtered, "Y": Y_train, "testX": X_test_filtered, "testY": Y_test
}
trainAndPredictDecisionTree(**parameters, value="pearson", max_depth=None)

pca = PCA(n_components=len(pearson_filtered_columns))
pca.fit(X_train_standardized)
X_train_pca = pca.transform(X_train_standardized)
X_test_pca = pca.transform(X_test_standardized)
parameters |= {
    "algorithm": "Decision Tree",
    "X": X_train_pca, "Y": Y_train, "testX": X_test_pca, "testY": Y_test
}
trainAndPredictDecisionTree(**parameters, value="pca", max_depth=None)

table.tableFooter()


|Algorithm| Hyperparameter | Value | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- | --- |
| Decision Tree | feature selection | baseline | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
|  |  | pearson | 0.9415 | 0.9419 | 0.9415 | 0.9412 |
|  |  | pca | 0.9357 | 0.9364 | 0.9357 | 0.9352 |


#### PCA vs Feature Selection Evaluation

PCA feature selection performed worse than Pearson or the baseline tree. While we saw that the PCA dimensions do encode the same information, this hints that the existing dimensions encode the information more tightly than PCA does. If there are "unused" features in the original dataset, PCA will use them for data. If those dimensions are then dropped, that will reduce the quality of the model.

Once the data is transformed using PCA, it is no longer interpretable. It is very difficult to take a path through the tree and match it back to features in the original dataset, since each PCA feature is multiple original features.

## 5. K-Fold Cross Validation

In [20]:
def report_score(score=None, score_accum=None):
    scores = score_accum.copy()
    scores = pd.concat([scores, score_accum.mean().rename("**Mean**").to_frame().T])
    scores = pd.concat([scores, score_accum.std().rename("**Std Dev**").to_frame().T])

    report(f"**{score}**")
    report(scores.to_markdown(floatfmt=".6f"))
    report("---")
    # report(score_accum.mean().rename("**Mean**").to_frame().T.to_markdown())
    # report(score_accum.std().rename("**Std Dev**").to_frame().T.to_markdown())

ignored = Table()

# Parameters shared by all callers.
parameters = {
  "cv":10,
  "table":ignored,
  "X":X_train, "Y":Y_train
}

report("**Step 1 - use normalised data**")
report("**Step 2 - create f1 and accuracy accumulators**")
f1_accum = pd.DataFrame()
accuracy_accum = pd.DataFrame()

# Setup for KNN
report("**Step 3 - run knn using cross_validate**")
parameters |= {
    "algorithm":"KNN", "hyperparameter":"n_neighbours"
}
knn_neighbours = [3,9]
for n in knn_neighbours:
    knn_scores = crossValidateUsingKnn(**parameters, value=n, n_neighbours=n)
    column_name = f"{parameters["algorithm"]}<br>(k={n})"
    accuracy_accum[column_name] = knn_scores["test_accuracy"]
    f1_accum[column_name] = knn_scores["test_f1_score"]

# Setup for Decision Tree
report("**Step 4 - run decision tree using cross_validate**")
parameters |= {
   "algorithm":"Decision Tree", "hyperparameter":"max_depth",
}
decision_tree_depths = [2,8]
for n in decision_tree_depths:
    dt_scores = crossValidateUsingTree(**parameters, value=n, max_depth=n)
    column_name = f"{parameters["algorithm"]}<br>(max_depth={n})"
    accuracy_accum[column_name] = dt_scores["test_accuracy"]
    f1_accum[column_name] = dt_scores["test_f1_score"]

report("**Step 5 - produce report**")
report_score("Accuracy", accuracy_accum)
report_score("F1 Score", f1_accum)



**Step 1 - use normalised data**

**Step 2 - create f1 and accuracy accumulators**

**Step 3 - run knn using cross_validate**

**Step 4 - run decision tree using cross_validate**

**Step 5 - produce report**

...

|             |   KNN<br>(k=3) |   KNN<br>(k=9) |   Decision Tree<br>(max_depth=2) |   Decision Tree<br>(max_depth=8) |
|:------------|---------------:|---------------:|---------------------------------:|---------------------------------:|
| 0           |       0.950000 |       0.925000 |                         0.850000 |                         0.825000 |
| 1           |       0.900000 |       0.925000 |                         0.800000 |                         0.875000 |
| 2           |       0.975000 |       0.975000 |                         0.900000 |                         0.950000 |
| 3           |       0.950000 |       0.950000 |                         0.950000 |                         0.900000 |
| 4           |       1.000000 |       0.975000 |                         0.875000 |                         0.925000 |
| 5           |       0.950000 |       0.975000 |                         0.950000 |                         0.975000 |
| 6           |       0.975000 |       0.975000 |                         0.925000 |                         0.925000 |
| 7           |       0.975000 |       0.950000 |                         0.900000 |                         0.950000 |
| 8           |       0.974359 |       0.974359 |                         0.897436 |                         1.000000 |
| 9           |       0.948718 |       1.000000 |                         0.948718 |                         0.923077 |
| **Mean**    |       0.959808 |       0.962436 |                         0.899615 |                         0.924808 |
| **Std Dev** |       0.026891 |       0.024260 |                         0.048452 |                         0.050004 |

---

**F1 Score**

|             |   KNN<br>(k=3) |   KNN<br>(k=9) |   Decision Tree<br>(max_depth=2) |   Decision Tree<br>(max_depth=8) |
|:------------|---------------:|---------------:|---------------------------------:|---------------------------------:|
| 0           |       0.950000 |       0.922545 |                         0.847009 |                         0.826302 |
| 1           |       0.898006 |       0.922545 |                         0.796011 |                         0.870909 |
| 2           |       0.974773 |       0.974773 |                         0.895238 |                         0.949003 |
| 3           |       0.949176 |       0.949176 |                         0.950000 |                         0.900000 |
| 4           |       1.000000 |       0.974814 |                         0.874070 |                         0.925444 |
| 5           |       0.950000 |       0.974814 |                         0.949176 |                         0.974814 |
| 6           |       0.974814 |       0.975148 |                         0.925444 |                         0.925444 |
| 7           |       0.974814 |       0.949176 |                         0.900000 |                         0.950000 |
| 8           |       0.974539 |       0.974539 |                         0.899387 |                         1.000000 |
| 9           |       0.948718 |       1.000000 |                         0.949359 |                         0.923618 |
| **Mean**    |       0.959484 |       0.961753 |                         0.898569 |                         0.924553 |
| **Std Dev** |       0.027393 |       0.025182 |                         0.049776 |                         0.050111 |

---

### Discuss Model Stablility and Reliability

KNN had a much more consistent result, with a standard deviation of 2.5 at K=9. The Decision Tree, with a standard deviation of ~5, had a much wider spread of results. This hints that KNN will generalize better than the Decision Tree.